RS_FinalVersion_Jan2026.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1c-hPCdVkMvOg_eTD-4guthb6BfoF7iPU

# Method 1- First Paper

# **Step 1**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV
from sklearn.metrics import (
    make_scorer,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler

In [ ]:
# 1️⃣ Load data
data = pd.read_csv("/content/TrainingSession_Etezadi.csv").dropna()
print(data.shape)

In [ ]:
# 2️⃣ Define target and independent variables (excluding soil features)
y = np.log10(data["mean_Om_p"])
independent_vars = [
    "BI_mean",
    "CI_mean",
    "NDMI_mean",
    "OMI_mean",
    "RI_mean",
    "SI_mean",
    "EVI_mean",
    "SAVI_mean",
    "NDVI_mean",
    "BSI_mean",
    "CAI_mean",
    "BI_stdDev",
    "CI_stdDev",
    "NDMI_stdDev",
    "OMI_stdDev",
    "RI_stdDev",
    "SI_stdDev",
    "EVI_stdDev",
    "SAVI_stdDev",
    "NDVI_stdDev",
    "BSI_stdDev",
    "CAI_stdDev",
]
X = data[independent_vars]

In [ ]:
# 3️⃣ Train-test split based on Field_no
unique_fields = data["Field_no"].unique()
np.random.seed(42)
test_fields = np.random.choice(
    unique_fields, size=int(0.2 * len(unique_fields)), replace=False
)
train_fields = np.setdiff1d(unique_fields, test_fields)

In [ ]:
train_data = data[data["Field_no"].isin(train_fields)]
test_data = data[data["Field_no"].isin(test_fields)]

In [ ]:
X_train = train_data[independent_vars]
y_train = np.log10(train_data["mean_Om_p"])
X_test = test_data[independent_vars]
y_test = np.log10(test_data["mean_Om_p"])

In [ ]:
# 4️⃣ Standard scaling (only once)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# 5️⃣ Feature Selection with LassoCV
lasso_cv = LassoCV(cv=5, random_state=42, max_iter=100000)
lasso_cv.fit(X_train_scaled, y_train)

In [ ]:
# Extract selected features
lasso_coefficients = pd.Series(lasso_cv.coef_, index=independent_vars)
important_features = lasso_coefficients[lasso_coefficients != 0].index.tolist()
print("Selected Features:", important_features)

In [ ]:
# 6️⃣ Rank images using Lasso CV-RMSE per image
image_rmse_records = []
kf = KFold(n_splits=3, shuffle=True, random_state=42)
rmse_scorer = make_scorer(mean_squared_error, squared=False)

In [ ]:
for img_id, df_img in train_data.groupby("Image_id"):
    if len(df_img) < 10:
        continue

    X_img = df_img[independent_vars]
    y_img = np.log10(df_img["mean_Om_p"])

    scaler_img = StandardScaler()
    X_img_scaled = scaler_img.fit_transform(X_img)

    lasso_model = LassoCV(cv=3, random_state=42, max_iter=10000)
    scores = cross_val_score(
        lasso_model, X_img_scaled, y_img, cv=kf, scoring=rmse_scorer
    )
    rmse_cv_mean = np.mean(scores)

    image_rmse_records.append({"Image_id": img_id, "RMSE": rmse_cv_mean})

In [ ]:
ranked_images_train = (
    pd.DataFrame(image_rmse_records).sort_values(by="RMSE").reset_index(drop=True)
)

In [ ]:
# 7️⃣ Progressive training with RF model
results = []
for i in range(1, len(ranked_images_train) + 1):
    selected_image_ids = ranked_images_train.iloc[:i]["Image_id"].tolist()
    scenario_data = train_data[train_data["Image_id"].isin(selected_image_ids)]

    X_train_scenario_raw = scenario_data[independent_vars]
    y_train_scenario = np.log10(scenario_data["mean_Om_p"])

    # Transform and reduce to important features
    X_train_scenario_scaled = scaler.transform(X_train_scenario_raw)
    X_train_scenario_scaled_selected = pd.DataFrame(
        X_train_scenario_scaled, columns=independent_vars
    )[important_features].values

    rf_model = RandomForestRegressor(random_state=42)
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(
        rf_model,
        X_train_scenario_scaled_selected,
        y_train_scenario,
        cv=kfold,
        scoring="neg_mean_squared_error",
    )
    cv_rmse_scores = np.sqrt(-cv_scores)

    rf_model.fit(X_train_scenario_scaled_selected, y_train_scenario)
    y_pred_scenario = rf_model.predict(
        X_test_scaled[:, [independent_vars.index(f) for f in important_features]]
    )

    mae = mean_absolute_error(y_test, y_pred_scenario)
    r2 = r2_score(y_test, y_pred_scenario)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_scenario))

    results.append(
        {
            "Iteration": i,
            "Images_Used": i,
            "MAE": mae,
            "R2": r2,
            "RMSE": rmse,
            "CV_RMSE_Mean": cv_rmse_scores.mean(),
            "CV_RMSE_Std": cv_rmse_scores.std(),
        }
    )

In [ ]:
# 8️⃣ Show results
results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
# 9️⃣ Plot results
fig, ax1 = plt.subplots(figsize=(12, 6))

In [ ]:
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="RMSE",
    marker="o",
    label="RMSE (Test)",
    color="blue",
    ax=ax1,
)
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="MAE",
    marker="s",
    label="MAE (Test)",
    color="green",
    ax=ax1,
)
ax1.set_xlabel("Number of Images Used")
ax1.set_ylabel("RMSE & MAE")
ax1.legend(loc="upper left")

In [ ]:
ax2 = ax1.twinx()
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="R2",
    marker="^",
    label="R² (Test)",
    color="red",
    ax=ax2,
)
ax2.set_ylabel("R² Score")
ax2.legend(loc="upper right")

In [ ]:
plt.title("Performance Metrics vs. Number of Images Used")
plt.grid()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.signal import savgol_filter
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV
from sklearn.metrics import (
    make_scorer,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler

In [ ]:
# 1️⃣ Load data
data = pd.read_csv("/content/TrainingSession_Etezadi.csv").dropna()
print(data.shape)

In [ ]:
# 2️⃣ Define target and independent variables (excluding soil features)
independent_vars = [
    "BI_mean",
    "CI_mean",
    "NDMI_mean",
    "OMI_mean",
    "RI_mean",
    "SI_mean",
    "EVI_mean",
    "SAVI_mean",
    "NDVI_mean",
    "BSI_mean",
    "CAI_mean",
    "BI_stdDev",
    "CI_stdDev",
    "NDMI_stdDev",
    "OMI_stdDev",
    "RI_stdDev",
    "SI_stdDev",
    "EVI_stdDev",
    "SAVI_stdDev",
    "NDVI_stdDev",
    "BSI_stdDev",
    "CAI_stdDev",
]

In [ ]:
y = np.log10(data["mean_Om_p"])
X = data[independent_vars]

In [ ]:
# 3️⃣ Train-test split based on Field_no
unique_fields = data["Field_no"].unique()
np.random.seed(42)
test_fields = np.random.choice(
    unique_fields, size=int(0.2 * len(unique_fields)), replace=False
)
train_fields = np.setdiff1d(unique_fields, test_fields)

In [ ]:
train_data = data[data["Field_no"].isin(train_fields)]
test_data = data[data["Field_no"].isin(test_fields)]

In [ ]:
X_test = test_data[independent_vars]
y_test = np.log10(test_data["mean_Om_p"])

In [ ]:
# 4️⃣ Feature Selection with LassoCV
scaler_full = StandardScaler()
X_train_scaled_full = scaler_full.fit_transform(train_data[independent_vars])
y_train_full = np.log10(train_data["mean_Om_p"])

In [ ]:
lasso_cv = LassoCV(cv=5, random_state=42, max_iter=100000)
lasso_cv.fit(X_train_scaled_full, y_train_full)

In [ ]:
lasso_coefficients = pd.Series(lasso_cv.coef_, index=independent_vars)
important_features = lasso_coefficients[lasso_coefficients != 0].index.tolist()
print("Selected Features:", important_features)

In [ ]:
# 5️⃣ Rank images using Lasso CV-RMSE per image
image_rmse_records = []
kf = KFold(n_splits=3, shuffle=True, random_state=42)
rmse_scorer = make_scorer(mean_squared_error, squared=False)

In [ ]:
for img_id, df_img in train_data.groupby("Image_id"):
    if len(df_img) < 10:
        continue

    X_img = df_img[independent_vars]
    y_img = np.log10(df_img["mean_Om_p"])

    scaler_img = StandardScaler()
    X_img_scaled = scaler_img.fit_transform(X_img)

    lasso_model = LassoCV(cv=3, random_state=42, max_iter=10000)
    scores = cross_val_score(
        lasso_model, X_img_scaled, y_img, cv=kf, scoring=rmse_scorer
    )
    rmse_cv_mean = np.mean(scores)

    image_rmse_records.append({"Image_id": img_id, "RMSE": rmse_cv_mean})

In [ ]:
ranked_images_train = (
    pd.DataFrame(image_rmse_records).sort_values(by="RMSE").reset_index(drop=True)
)

In [ ]:
# 🔍 Automatically find best threshold to remove bad images
def auto_threshold_selection(r2_track, max_threshold=0.05, step=0.001):
    best_threshold = 0
    best_r2 = -np.inf
    thresholds = np.arange(0.001, max_threshold, step)
    for thresh in thresholds:
        r2_copy = r2_track.copy()
        drop_indices = [
            i - 1
            for i in range(2, len(r2_copy))
            if r2_copy[i] < r2_copy[i - 2] - thresh
        ]
        r2_masked = [r2_copy[i] for i in range(len(r2_copy)) if i not in drop_indices]
        avg_r2 = np.mean(r2_masked) if r2_masked else -np.inf
        if avg_r2 > best_r2:
            best_r2 = avg_r2
            best_threshold = thresh
    return best_threshold

In [ ]:
# Dummy run to get R2 values
r2_track = []
for i in range(1, len(ranked_images_train) + 1):
    selected_image_ids = ranked_images_train.iloc[:i]["Image_id"].tolist()
    scenario_data = train_data[train_data["Image_id"].isin(selected_image_ids)]
    X_train_scenario_raw = scenario_data[independent_vars]
    y_train_scenario = np.log10(scenario_data["mean_Om_p"])
    scaler_scenario = StandardScaler()
    X_train_scenario_scaled = scaler_scenario.fit_transform(X_train_scenario_raw)
    X_train_scenario_scaled_selected = pd.DataFrame(
        X_train_scenario_scaled, columns=independent_vars
    )[important_features].values
    X_test_scaled = scaler_scenario.transform(X_test)
    X_test_scaled_selected = pd.DataFrame(X_test_scaled, columns=independent_vars)[
        important_features
    ].values
    rf_model = RandomForestRegressor(random_state=42)
    rf_model.fit(X_train_scenario_scaled_selected, y_train_scenario)
    y_pred = rf_model.predict(X_test_scaled_selected)
    r2 = r2_score(y_test, y_pred)
    r2_track.append(r2)

In [ ]:
# Select best threshold and remove bad images
r2_drop_threshold = auto_threshold_selection(r2_track)
bad_iterations = [
    i - 1
    for i in range(2, len(r2_track))
    if r2_track[i] < r2_track[i - 2] - r2_drop_threshold
]
bad_images = ranked_images_train.iloc[bad_iterations]["Image_id"].tolist()
ranked_images_train = ranked_images_train[
    ~ranked_images_train["Image_id"].isin(bad_images)
].reset_index(drop=True)

In [ ]:
# 6️⃣ Progressive training with RF model (cleaned)
results = []
for i in range(1, len(ranked_images_train) + 1):
    selected_image_ids = ranked_images_train.iloc[:i]["Image_id"].tolist()
    scenario_data = train_data[train_data["Image_id"].isin(selected_image_ids)]

    X_train_scenario_raw = scenario_data[independent_vars]
    y_train_scenario = np.log10(scenario_data["mean_Om_p"])

    scaler_scenario = StandardScaler()
    X_train_scenario_scaled = scaler_scenario.fit_transform(X_train_scenario_raw)
    X_train_scenario_scaled_selected = pd.DataFrame(
        X_train_scenario_scaled, columns=independent_vars
    )[important_features].values

    X_test_scenario_scaled = scaler_scenario.transform(X_test)
    X_test_scenario_scaled_selected = pd.DataFrame(
        X_test_scenario_scaled, columns=independent_vars
    )[important_features].values

    rf_model = RandomForestRegressor(random_state=42)
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(
        rf_model,
        X_train_scenario_scaled_selected,
        y_train_scenario,
        cv=kfold,
        scoring="neg_mean_squared_error",
    )
    cv_rmse_scores = np.sqrt(-cv_scores)

    rf_model.fit(X_train_scenario_scaled_selected, y_train_scenario)
    y_pred_scenario = rf_model.predict(X_test_scenario_scaled_selected)

    mae = mean_absolute_error(y_test, y_pred_scenario)
    r2 = r2_score(y_test, y_pred_scenario)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_scenario))

    results.append(
        {
            "Iteration": i,
            "Images_Used": i,
            "MAE": mae,
            "R2": r2,
            "RMSE": rmse,
            "CV_RMSE_Mean": cv_rmse_scores.mean(),
            "CV_RMSE_Std": cv_rmse_scores.std(),
        }
    )

In [ ]:
results_df = pd.DataFrame(results)

In [ ]:
# 🔁 Smoothing R2 using multiple methods
results_df["R2_smooth"] = results_df["R2"].rolling(window=3, center=True).mean()
results_df["R2_ewm"] = results_df["R2"].ewm(span=3, adjust=False).mean()
results_df["R2_savgol"] = savgol_filter(
    results_df["R2"],
    window_length=(
        5 if len(results_df) >= 5 else len(results_df) - (len(results_df) % 2 == 0)
    ),
    polyorder=2,
)

In [ ]:
# 📊 Final Plot
fig, ax1 = plt.subplots(figsize=(12, 6))
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="RMSE",
    marker="o",
    label="RMSE (Test)",
    color="blue",
    ax=ax1,
)
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="MAE",
    marker="s",
    label="MAE (Test)",
    color="orange",
    ax=ax1,
)
ax1.set_xlabel("Number of Images Used")
ax1.set_ylabel("RMSE & MAE")
ax1.legend(loc="upper left")

In [ ]:
ax2 = ax1.twinx()
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="R2_smooth",
    marker="^",
    label="R² (Rolling)",
    color="red",
    ax=ax2,
)
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="R2_ewm",
    marker="x",
    label="R² (EWMA)",
    color="green",
    ax=ax2,
)
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="R2_savgol",
    marker="D",
    label="R² (Savitzky-Golay)",
    color="purple",
    ax=ax2,
)
ax2.set_ylabel("R² Score")
ax2.legend(loc="upper right")

In [ ]:
plt.title("Performance Metrics vs. Number of Images Used (After Removing Bad Images)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LassoCV
from sklearn.metrics import (
    make_scorer,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler

In [ ]:
# 1️⃣ Load data
data = pd.read_csv("/content/TrainingSession_Etezadi.csv").dropna()
print(data.shape)

In [ ]:
# 2️⃣ Convert soilTypes column to binary features
data["soilTypes"] = (
    data["soilTypes"].str.strip("[]").str.replace("'", "").str.split(", ")
)
vectorizer = CountVectorizer(tokenizer=lambda x: x, lowercase=False, preprocessor=None)
soil_features = vectorizer.fit_transform(data["soilTypes"]).toarray()
soil_feature_names = vectorizer.get_feature_names_out()
soil_features_df = pd.DataFrame(soil_features, columns=soil_feature_names)
data = pd.concat([data.reset_index(drop=True), soil_features_df], axis=1)

In [ ]:
# 3️⃣ Define target and independent variables
soil_vars = list(soil_features_df.columns)
independent_vars = [
    "BI_mean",
    "CI_mean",
    "NDMI_mean",
    "OMI_mean",
    "RI_mean",
    "SI_mean",
    "EVI_mean",
    "SAVI_mean",
    "NDVI_mean",
    "BSI_mean",
    "CAI_mean",
    "BI_stdDev",
    "CI_stdDev",
    "NDMI_stdDev",
    "OMI_stdDev",
    "RI_stdDev",
    "SI_stdDev",
    "EVI_stdDev",
    "SAVI_stdDev",
    "NDVI_stdDev",
    "BSI_stdDev",
    "CAI_stdDev",
] + soil_vars

In [ ]:
y = np.log10(data["mean_Om_p"])
X = data[independent_vars]

In [ ]:
# 4️⃣ Train-test split based on Field_no
unique_fields = data["Field_no"].unique()
np.random.seed(42)
test_fields = np.random.choice(
    unique_fields, size=int(0.2 * len(unique_fields)), replace=False
)
train_fields = np.setdiff1d(unique_fields, test_fields)

In [ ]:
train_data = data[data["Field_no"].isin(train_fields)]
test_data = data[data["Field_no"].isin(test_fields)]

In [ ]:
X_train = train_data[independent_vars]
y_train = np.log10(train_data["mean_Om_p"])
X_test = test_data[independent_vars]
y_test = np.log10(test_data["mean_Om_p"])

In [ ]:
# 5️⃣ Standard scaling (only once)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# 6️⃣ Feature Selection with LassoCV
lasso_cv = LassoCV(cv=5, random_state=42, max_iter=100000)
lasso_cv.fit(X_train_scaled, y_train)

In [ ]:
# Extract selected features
lasso_coefficients = pd.Series(lasso_cv.coef_, index=independent_vars)
important_features = lasso_coefficients[lasso_coefficients != 0].index.tolist()
print("Selected Features:", important_features)

In [ ]:
# 7️⃣ Rank images using Lasso CV-RMSE per image
image_rmse_records = []
kf = KFold(n_splits=3, shuffle=True, random_state=42)
rmse_scorer = make_scorer(mean_squared_error, squared=False)

In [ ]:
for img_id, df_img in train_data.groupby("Image_id"):
    if len(df_img) < 10:
        continue

    X_img = df_img[independent_vars]
    y_img = np.log10(df_img["mean_Om_p"])

    scaler_img = StandardScaler()
    X_img_scaled = scaler_img.fit_transform(X_img)

    lasso_model = LassoCV(cv=3, random_state=42, max_iter=10000)
    scores = cross_val_score(
        lasso_model, X_img_scaled, y_img, cv=kf, scoring=rmse_scorer
    )
    rmse_cv_mean = np.mean(scores)

    image_rmse_records.append({"Image_id": img_id, "RMSE": rmse_cv_mean})

In [ ]:
ranked_images_train = (
    pd.DataFrame(image_rmse_records).sort_values(by="RMSE").reset_index(drop=True)
)

In [ ]:
# 8️⃣ Progressive training with RF model
results = []
for i in range(1, len(ranked_images_train) + 1):
    selected_image_ids = ranked_images_train.iloc[:i]["Image_id"].tolist()
    scenario_data = train_data[train_data["Image_id"].isin(selected_image_ids)]

    X_train_scenario_raw = scenario_data[independent_vars]
    y_train_scenario = np.log10(scenario_data["mean_Om_p"])

    # Transform and reduce to important features
    X_train_scenario_scaled = scaler.transform(X_train_scenario_raw)
    X_train_scenario_scaled_selected = pd.DataFrame(
        X_train_scenario_scaled, columns=independent_vars
    )[important_features].values

    rf_model = RandomForestRegressor(random_state=42)
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(
        rf_model,
        X_train_scenario_scaled_selected,
        y_train_scenario,
        cv=kfold,
        scoring="neg_mean_squared_error",
    )
    cv_rmse_scores = np.sqrt(-cv_scores)

    rf_model.fit(X_train_scenario_scaled_selected, y_train_scenario)
    y_pred_scenario = rf_model.predict(
        X_test_scaled[:, [independent_vars.index(f) for f in important_features]]
    )

    mae = mean_absolute_error(y_test, y_pred_scenario)
    r2 = r2_score(y_test, y_pred_scenario)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_scenario))

    results.append(
        {
            "Iteration": i,
            "Images_Used": i,
            "MAE": mae,
            "R2": r2,
            "RMSE": rmse,
            "CV_RMSE_Mean": cv_rmse_scores.mean(),
            "CV_RMSE_Std": cv_rmse_scores.std(),
        }
    )

In [ ]:
# 9️⃣ Show results
results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
# 🔟 Plot results
fig, ax1 = plt.subplots(figsize=(12, 6))

In [ ]:
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="RMSE",
    marker="o",
    label="RMSE (Test)",
    color="blue",
    ax=ax1,
)
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="MAE",
    marker="s",
    label="MAE (Test)",
    color="green",
    ax=ax1,
)
ax1.set_xlabel("Number of Images Used")
ax1.set_ylabel("RMSE & MAE")
ax1.legend(loc="upper left")

In [ ]:
ax2 = ax1.twinx()
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="R2",
    marker="^",
    label="R² (Test)",
    color="red",
    ax=ax2,
)
ax2.set_ylabel("R² Score")
ax2.legend(loc="upper right")

In [ ]:
plt.title("Performance Metrics vs. Number of Images Used")
plt.grid()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LassoCV
from sklearn.metrics import (
    make_scorer,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler

In [ ]:
# 1️⃣ Load data
data = pd.read_csv("/content/TrainingSession_Etezadi.csv").dropna()
print(data.shape)

In [ ]:
# 2️⃣ Convert soilTypes column to binary features
data["soilTypes"] = (
    data["soilTypes"].str.strip("[]").str.replace("'", "").str.split(", ")
)
vectorizer = CountVectorizer(tokenizer=lambda x: x, lowercase=False, preprocessor=None)
soil_features = vectorizer.fit_transform(data["soilTypes"]).toarray()
soil_feature_names = vectorizer.get_feature_names_out()
soil_features_df = pd.DataFrame(soil_features, columns=soil_feature_names)
data = pd.concat([data.reset_index(drop=True), soil_features_df], axis=1)

In [ ]:
# 3️⃣ Define target and independent variables
soil_vars = list(soil_features_df.columns)
independent_vars = [
    "BI_mean",
    "CI_mean",
    "NDMI_mean",
    "OMI_mean",
    "RI_mean",
    "SI_mean",
    "EVI_mean",
    "SAVI_mean",
    "NDVI_mean",
    "BSI_mean",
    "CAI_mean",
    "BI_stdDev",
    "CI_stdDev",
    "NDMI_stdDev",
    "OMI_stdDev",
    "RI_stdDev",
    "SI_stdDev",
    "EVI_stdDev",
    "SAVI_stdDev",
    "NDVI_stdDev",
    "BSI_stdDev",
    "CAI_stdDev",
] + soil_vars

In [ ]:
y = np.log10(data["mean_Om_p"])
X = data[independent_vars]

In [ ]:
# 4️⃣ Train-test split based on Field_no
unique_fields = data["Field_no"].unique()
np.random.seed(42)
test_fields = np.random.choice(
    unique_fields, size=int(0.2 * len(unique_fields)), replace=False
)
train_fields = np.setdiff1d(unique_fields, test_fields)

In [ ]:
train_data = data[data["Field_no"].isin(train_fields)]
test_data = data[data["Field_no"].isin(test_fields)]

In [ ]:
X_train = train_data[independent_vars]
y_train = np.log10(train_data["mean_Om_p"])
X_test = test_data[independent_vars]
y_test = np.log10(test_data["mean_Om_p"])

In [ ]:
# 5️⃣ Standard scaling (only once)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# 6️⃣ Feature Selection with LassoCV
lasso_cv = LassoCV(cv=5, random_state=42, max_iter=100000)
lasso_cv.fit(X_train_scaled, y_train)

In [ ]:
# Extract selected features
lasso_coefficients = pd.Series(lasso_cv.coef_, index=independent_vars)
important_features = lasso_coefficients[lasso_coefficients != 0].index.tolist()
print("Selected Features:", important_features)

In [ ]:
# 7️⃣ Rank images using Lasso CV-RMSE per image
image_rmse_records = []
kf = KFold(n_splits=3, shuffle=True, random_state=42)
rmse_scorer = make_scorer(mean_squared_error, squared=False)

In [ ]:
for img_id, df_img in train_data.groupby("Image_id"):
    if len(df_img) < 10:
        continue

    X_img = df_img[independent_vars]
    y_img = np.log10(df_img["mean_Om_p"])

    scaler_img = StandardScaler()
    X_img_scaled = scaler_img.fit_transform(X_img)

    lasso_model = LassoCV(cv=3, random_state=42, max_iter=10000)
    scores = cross_val_score(
        lasso_model, X_img_scaled, y_img, cv=kf, scoring=rmse_scorer
    )
    rmse_cv_mean = np.mean(scores)

    image_rmse_records.append({"Image_id": img_id, "RMSE": rmse_cv_mean})

In [ ]:
ranked_images_train = (
    pd.DataFrame(image_rmse_records).sort_values(by="RMSE").reset_index(drop=True)
)

In [ ]:
# 8️⃣ Progressive training with RF model
results = []
for i in range(1, len(ranked_images_train) + 1):
    selected_image_ids = ranked_images_train.iloc[:i]["Image_id"].tolist()
    scenario_data = train_data[train_data["Image_id"].isin(selected_image_ids)]

    X_train_scenario_raw = scenario_data[independent_vars]
    y_train_scenario = np.log10(scenario_data["mean_Om_p"])

    X_train_scenario_scaled = scaler.transform(X_train_scenario_raw)
    X_train_scenario_scaled_selected = pd.DataFrame(
        X_train_scenario_scaled, columns=independent_vars
    )[important_features].values

    rf_model = RandomForestRegressor(random_state=42)
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(
        rf_model,
        X_train_scenario_scaled_selected,
        y_train_scenario,
        cv=kfold,
        scoring="neg_mean_squared_error",
    )
    cv_rmse_scores = np.sqrt(-cv_scores)

    rf_model.fit(X_train_scenario_scaled_selected, y_train_scenario)
    y_pred_scenario = rf_model.predict(
        X_test_scaled[:, [independent_vars.index(f) for f in important_features]]
    )

    mae = mean_absolute_error(y_test, y_pred_scenario)
    r2 = r2_score(y_test, y_pred_scenario)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_scenario))

    results.append(
        {
            "Iteration": i,
            "Images_Used": i,
            "MAE": mae,
            "R2": r2,
            "RMSE": rmse,
            "CV_RMSE_Mean": cv_rmse_scores.mean(),
            "CV_RMSE_Std": cv_rmse_scores.std(),
        }
    )

In [ ]:
results_df = pd.DataFrame(results)

In [ ]:
# 🔁 Identify and remove bad images based on R2 drop threshold
drop_threshold = 0.01
bad_iterations = []
for i in range(1, len(results_df)):
    if results_df.loc[i, "R2"] < results_df.loc[i - 1, "R2"] - drop_threshold:
        bad_iterations.append(i)

In [ ]:
bad_images = ranked_images_train.iloc[bad_iterations]["Image_id"].tolist()
print("Bad images detected:", bad_images)

In [ ]:
# Remove bad images from ranking
ranked_images_train = ranked_images_train[
    ~ranked_images_train["Image_id"].isin(bad_images)
].reset_index(drop=True)

In [ ]:
# 🔁 Re-run RF training on good images only
results = []
for i in range(1, len(ranked_images_train) + 1):
    selected_image_ids = ranked_images_train.iloc[:i]["Image_id"].tolist()
    scenario_data = train_data[train_data["Image_id"].isin(selected_image_ids)]

    X_train_scenario_raw = scenario_data[independent_vars]
    y_train_scenario = np.log10(scenario_data["mean_Om_p"])

    X_train_scenario_scaled = scaler.transform(X_train_scenario_raw)
    X_train_scenario_scaled_selected = pd.DataFrame(
        X_train_scenario_scaled, columns=independent_vars
    )[important_features].values

    rf_model = RandomForestRegressor(random_state=42)
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(
        rf_model,
        X_train_scenario_scaled_selected,
        y_train_scenario,
        cv=kfold,
        scoring="neg_mean_squared_error",
    )
    cv_rmse_scores = np.sqrt(-cv_scores)

    rf_model.fit(X_train_scenario_scaled_selected, y_train_scenario)
    y_pred_scenario = rf_model.predict(
        X_test_scaled[:, [independent_vars.index(f) for f in important_features]]
    )

    mae = mean_absolute_error(y_test, y_pred_scenario)
    r2 = r2_score(y_test, y_pred_scenario)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_scenario))

    results.append(
        {
            "Iteration": i,
            "Images_Used": i,
            "MAE": mae,
            "R2": r2,
            "RMSE": rmse,
            "CV_RMSE_Mean": cv_rmse_scores.mean(),
            "CV_RMSE_Std": cv_rmse_scores.std(),
        }
    )

In [ ]:
# 📊 Final results and plot
results_df = pd.DataFrame(results)
results_df["R2_smooth"] = results_df["R2"].rolling(window=3, center=True).mean()

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="RMSE",
    marker="o",
    label="RMSE (Test)",
    color="blue",
    ax=ax1,
)
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="MAE",
    marker="s",
    label="MAE (Test)",
    color="orange",
    ax=ax1,
)
ax1.set_xlabel("Number of Images Used")
ax1.set_ylabel("RMSE & MAE")
ax1.legend(loc="upper left")

In [ ]:
ax2 = ax1.twinx()
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="R2",
    marker="^",
    label="R² (Test)",
    color="red",
    ax=ax2,
)
ax2.set_ylabel("R² Score")
ax2.legend(loc="upper right")

In [ ]:
plt.title("Performance Metrics vs. Number of Images Used (After Removing Drop Images)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LassoCV
from sklearn.metrics import (
    make_scorer,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler

In [ ]:
# 1️⃣ Load data
data = pd.read_csv("/content/TrainingSession_Etezadi.csv").dropna()
print(data.shape)

In [ ]:
# 2️⃣ Convert soilTypes column to binary features
data["soilTypes"] = (
    data["soilTypes"].str.strip("[]").str.replace("'", "").str.split(", ")
)
vectorizer = CountVectorizer(tokenizer=lambda x: x, lowercase=False, preprocessor=None)
soil_features = vectorizer.fit_transform(data["soilTypes"]).toarray()
soil_feature_names = vectorizer.get_feature_names_out()
soil_features_df = pd.DataFrame(soil_features, columns=soil_feature_names)
data = pd.concat([data.reset_index(drop=True), soil_features_df], axis=1)

In [ ]:
# 3️⃣ Define target and independent variables
soil_vars = list(soil_features_df.columns)
independent_vars = [
    "BI_mean",
    "CI_mean",
    "NDMI_mean",
    "OMI_mean",
    "RI_mean",
    "SI_mean",
    "EVI_mean",
    "SAVI_mean",
    "NDVI_mean",
    "BSI_mean",
    "CAI_mean",
    "BI_stdDev",
    "CI_stdDev",
    "NDMI_stdDev",
    "OMI_stdDev",
    "RI_stdDev",
    "SI_stdDev",
    "EVI_stdDev",
    "SAVI_stdDev",
    "NDVI_stdDev",
    "BSI_stdDev",
    "CAI_stdDev",
] + soil_vars

In [ ]:
y = np.log10(data["mean_Om_p"])
X = data[independent_vars]

In [ ]:
# 4️⃣ Train-test split based on Field_no
unique_fields = data["Field_no"].unique()
np.random.seed(42)
test_fields = np.random.choice(
    unique_fields, size=int(0.2 * len(unique_fields)), replace=False
)
train_fields = np.setdiff1d(unique_fields, test_fields)

In [ ]:
train_data = data[data["Field_no"].isin(train_fields)]
test_data = data[data["Field_no"].isin(test_fields)]

In [ ]:
X_train = train_data[independent_vars]
y_train = np.log10(train_data["mean_Om_p"])
X_test = test_data[independent_vars]
y_test = np.log10(test_data["mean_Om_p"])

In [ ]:
# 5️⃣ Standard scaling (only once)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# 6️⃣ Feature Selection with LassoCV
lasso_cv = LassoCV(cv=5, random_state=42, max_iter=100000)
lasso_cv.fit(X_train_scaled, y_train)

In [ ]:
# Extract selected features
lasso_coefficients = pd.Series(lasso_cv.coef_, index=independent_vars)
important_features = lasso_coefficients[lasso_coefficients != 0].index.tolist()
print("Selected Features:", important_features)

In [ ]:
# 7️⃣ Rank images using Lasso CV-RMSE per image
image_rmse_records = []
kf = KFold(n_splits=3, shuffle=True, random_state=42)
rmse_scorer = make_scorer(mean_squared_error, squared=False)

In [ ]:
for img_id, df_img in train_data.groupby("Image_id"):
    if len(df_img) < 10:
        continue

    X_img = df_img[independent_vars]
    y_img = np.log10(df_img["mean_Om_p"])

    scaler_img = StandardScaler()
    X_img_scaled = scaler_img.fit_transform(X_img)

    lasso_model = LassoCV(cv=3, random_state=42, max_iter=10000)
    scores = cross_val_score(
        lasso_model, X_img_scaled, y_img, cv=kf, scoring=rmse_scorer
    )
    rmse_cv_mean = np.mean(scores)

    image_rmse_records.append({"Image_id": img_id, "RMSE": rmse_cv_mean})

In [ ]:
ranked_images_train = (
    pd.DataFrame(image_rmse_records).sort_values(by="RMSE").reset_index(drop=True)
)

In [ ]:
# 8️⃣ Progressive training with RF model
results = []
for i in range(1, len(ranked_images_train) + 1):
    selected_image_ids = ranked_images_train.iloc[:i]["Image_id"].tolist()
    scenario_data = train_data[train_data["Image_id"].isin(selected_image_ids)]

    X_train_scenario_raw = scenario_data[independent_vars]
    y_train_scenario = np.log10(scenario_data["mean_Om_p"])

    X_train_scenario_scaled = scaler.transform(X_train_scenario_raw)
    X_train_scenario_scaled_selected = pd.DataFrame(
        X_train_scenario_scaled, columns=independent_vars
    )[important_features].values

    rf_model = RandomForestRegressor(random_state=42)
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(
        rf_model,
        X_train_scenario_scaled_selected,
        y_train_scenario,
        cv=kfold,
        scoring="neg_mean_squared_error",
    )
    cv_rmse_scores = np.sqrt(-cv_scores)

    rf_model.fit(X_train_scenario_scaled_selected, y_train_scenario)
    y_pred_scenario = rf_model.predict(
        X_test_scaled[:, [independent_vars.index(f) for f in important_features]]
    )

    mae = mean_absolute_error(y_test, y_pred_scenario)
    r2 = r2_score(y_test, y_pred_scenario)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_scenario))

    results.append(
        {
            "Iteration": i,
            "Images_Used": i,
            "MAE": mae,
            "R2": r2,
            "RMSE": rmse,
            "CV_RMSE_Mean": cv_rmse_scores.mean(),
            "CV_RMSE_Std": cv_rmse_scores.std(),
        }
    )

In [ ]:
results_df = pd.DataFrame(results)

In [ ]:
# 🔁 Identify and remove bad images based on R2 drop threshold
drop_threshold = 0.01
bad_iterations = []
for i in range(1, len(results_df)):
    if results_df.loc[i, "R2"] < results_df.loc[i - 1, "R2"] - drop_threshold:
        bad_iterations.append(i)

In [ ]:
bad_images = ranked_images_train.iloc[bad_iterations]["Image_id"].tolist()
print("Bad images detected:", bad_images)

In [ ]:
# Remove bad images from ranking
ranked_images_train = ranked_images_train[
    ~ranked_images_train["Image_id"].isin(bad_images)
].reset_index(drop=True)

In [ ]:
# 🔁 Re-run RF training on good images only
results = []
for i in range(1, len(ranked_images_train) + 1):
    selected_image_ids = ranked_images_train.iloc[:i]["Image_id"].tolist()
    scenario_data = train_data[train_data["Image_id"].isin(selected_image_ids)]

    X_train_scenario_raw = scenario_data[independent_vars]
    y_train_scenario = np.log10(scenario_data["mean_Om_p"])

    X_train_scenario_scaled = scaler.transform(X_train_scenario_raw)
    X_train_scenario_scaled_selected = pd.DataFrame(
        X_train_scenario_scaled, columns=independent_vars
    )[important_features].values

    rf_model = RandomForestRegressor(random_state=42)
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(
        rf_model,
        X_train_scenario_scaled_selected,
        y_train_scenario,
        cv=kfold,
        scoring="neg_mean_squared_error",
    )
    cv_rmse_scores = np.sqrt(-cv_scores)

    rf_model.fit(X_train_scenario_scaled_selected, y_train_scenario)
    y_pred_scenario = rf_model.predict(
        X_test_scaled[:, [independent_vars.index(f) for f in important_features]]
    )

    mae = mean_absolute_error(y_test, y_pred_scenario)
    r2 = r2_score(y_test, y_pred_scenario)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_scenario))

    results.append(
        {
            "Iteration": i,
            "Images_Used": i,
            "MAE": mae,
            "R2": r2,
            "RMSE": rmse,
            "CV_RMSE_Mean": cv_rmse_scores.mean(),
            "CV_RMSE_Std": cv_rmse_scores.std(),
        }
    )

In [ ]:
# 📊 Final results and plot
results_df = pd.DataFrame(results)
results_df["R2_smooth"] = results_df["R2"].rolling(window=3, center=True).mean()

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="RMSE",
    marker="o",
    label="RMSE (Test)",
    color="blue",
    ax=ax1,
)
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="MAE",
    marker="s",
    label="MAE (Test)",
    color="orange",
    ax=ax1,
)
ax1.set_xlabel("Number of Images Used")
ax1.set_ylabel("RMSE & MAE")
ax1.legend(loc="upper left")

In [ ]:
ax2 = ax1.twinx()
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="R2",
    marker="^",
    label="R² (Test)",
    color="red",
    ax=ax2,
)
ax2.set_ylabel("R² Score")
ax2.legend(loc="upper right")

In [ ]:
plt.title("Performance Metrics vs. Number of Images Used (After Removing Drop Images)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LassoCV
from sklearn.metrics import (
    make_scorer,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler

In [ ]:
# 1️⃣ Load data
data = pd.read_csv("/content/TrainingSession_Etezadi.csv").dropna()
print(data.shape)

In [ ]:
# 2️⃣ Convert soilTypes column to binary features
data["soilTypes"] = (
    data["soilTypes"].str.strip("[]").str.replace("'", "").str.split(", ")
)
vectorizer = CountVectorizer(tokenizer=lambda x: x, lowercase=False, preprocessor=None)
soil_features = vectorizer.fit_transform(data["soilTypes"]).toarray()
soil_feature_names = vectorizer.get_feature_names_out()
soil_features_df = pd.DataFrame(soil_features, columns=soil_feature_names)
data = pd.concat([data.reset_index(drop=True), soil_features_df], axis=1)

In [ ]:
# 3️⃣ Define target and independent variables
soil_vars = list(soil_features_df.columns)
independent_vars = [
    "BI_mean",
    "CI_mean",
    "NDMI_mean",
    "OMI_mean",
    "RI_mean",
    "SI_mean",
    "EVI_mean",
    "SAVI_mean",
    "NDVI_mean",
    "BSI_mean",
    "CAI_mean",
    "BI_stdDev",
    "CI_stdDev",
    "NDMI_stdDev",
    "OMI_stdDev",
    "RI_stdDev",
    "SI_stdDev",
    "EVI_stdDev",
    "SAVI_stdDev",
    "NDVI_stdDev",
    "BSI_stdDev",
    "CAI_stdDev",
] + soil_vars

In [ ]:
y = np.log10(data["mean_Om_p"])
X = data[independent_vars]

In [ ]:
# 4️⃣ Train-test split based on Field_no
unique_fields = data["Field_no"].unique()
np.random.seed(42)
test_fields = np.random.choice(
    unique_fields, size=int(0.2 * len(unique_fields)), replace=False
)
train_fields = np.setdiff1d(unique_fields, test_fields)

In [ ]:
train_data = data[data["Field_no"].isin(train_fields)]
test_data = data[data["Field_no"].isin(test_fields)]

In [ ]:
X_train = train_data[independent_vars]
y_train = np.log10(train_data["mean_Om_p"])
X_test = test_data[independent_vars]
y_test = np.log10(test_data["mean_Om_p"])

In [ ]:
# 5️⃣ Standard scaling (only once)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# 6️⃣ Feature Selection with LassoCV
lasso_cv = LassoCV(cv=5, random_state=42, max_iter=100000)
lasso_cv.fit(X_train_scaled, y_train)

In [ ]:
# Extract selected features
lasso_coefficients = pd.Series(lasso_cv.coef_, index=independent_vars)
important_features = lasso_coefficients[lasso_coefficients != 0].index.tolist()
print("Selected Features:", important_features)

In [ ]:
# 7️⃣ Rank images using Lasso CV-RMSE per image
image_rmse_records = []
kf = KFold(n_splits=3, shuffle=True, random_state=42)
rmse_scorer = make_scorer(mean_squared_error, squared=False)

In [ ]:
for img_id, df_img in train_data.groupby("Image_id"):
    if len(df_img) < 10:
        continue

    X_img = df_img[independent_vars]
    y_img = np.log10(df_img["mean_Om_p"])

    scaler_img = StandardScaler()
    X_img_scaled = scaler_img.fit_transform(X_img)

    lasso_model = LassoCV(cv=3, random_state=42, max_iter=10000)
    scores = cross_val_score(
        lasso_model, X_img_scaled, y_img, cv=kf, scoring=rmse_scorer
    )
    rmse_cv_mean = np.mean(scores)

    image_rmse_records.append({"Image_id": img_id, "RMSE": rmse_cv_mean})

In [ ]:
ranked_images_train = (
    pd.DataFrame(image_rmse_records).sort_values(by="RMSE").reset_index(drop=True)
)

In [ ]:
# 8️⃣ Progressive training with RF model
results = []
for i in range(1, len(ranked_images_train) + 1):
    selected_image_ids = ranked_images_train.iloc[:i]["Image_id"].tolist()
    scenario_data = train_data[train_data["Image_id"].isin(selected_image_ids)]

    X_train_scenario_raw = scenario_data[independent_vars]
    y_train_scenario = np.log10(scenario_data["mean_Om_p"])

    X_train_scenario_scaled = scaler.transform(X_train_scenario_raw)
    X_train_scenario_scaled_selected = pd.DataFrame(
        X_train_scenario_scaled, columns=independent_vars
    )[important_features].values

    rf_model = RandomForestRegressor(random_state=42)
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(
        rf_model,
        X_train_scenario_scaled_selected,
        y_train_scenario,
        cv=kfold,
        scoring="neg_mean_squared_error",
    )
    cv_rmse_scores = np.sqrt(-cv_scores)

    rf_model.fit(X_train_scenario_scaled_selected, y_train_scenario)
    y_pred_scenario = rf_model.predict(
        X_test_scaled[:, [independent_vars.index(f) for f in important_features]]
    )

    mae = mean_absolute_error(y_test, y_pred_scenario)
    r2 = r2_score(y_test, y_pred_scenario)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_scenario))

    results.append(
        {
            "Iteration": i,
            "Images_Used": i,
            "MAE": mae,
            "R2": r2,
            "RMSE": rmse,
            "CV_RMSE_Mean": cv_rmse_scores.mean(),
            "CV_RMSE_Std": cv_rmse_scores.std(),
        }
    )

In [ ]:
results_df = pd.DataFrame(results)

In [ ]:
# 🔁 Identify and remove bad images based on R2 drop threshold
drop_threshold = 0.01
bad_iterations = []
for i in range(1, len(results_df)):
    if results_df.loc[i, "R2"] < results_df.loc[i - 1, "R2"] - drop_threshold:
        bad_iterations.append(i)

In [ ]:
bad_images = ranked_images_train.iloc[bad_iterations]["Image_id"].tolist()
print("Bad images detected:", bad_images)

In [ ]:
# Remove bad images from ranking
ranked_images_train = ranked_images_train[
    ~ranked_images_train["Image_id"].isin(bad_images)
].reset_index(drop=True)

In [ ]:
# 🔁 Re-run RF training on good images only
results = []
for i in range(1, len(ranked_images_train) + 1):
    selected_image_ids = ranked_images_train.iloc[:i]["Image_id"].tolist()
    scenario_data = train_data[train_data["Image_id"].isin(selected_image_ids)]

    X_train_scenario_raw = scenario_data[independent_vars]
    y_train_scenario = np.log10(scenario_data["mean_Om_p"])

    X_train_scenario_scaled = scaler.transform(X_train_scenario_raw)
    X_train_scenario_scaled_selected = pd.DataFrame(
        X_train_scenario_scaled, columns=independent_vars
    )[important_features].values

    rf_model = RandomForestRegressor(random_state=42)
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(
        rf_model,
        X_train_scenario_scaled_selected,
        y_train_scenario,
        cv=kfold,
        scoring="neg_mean_squared_error",
    )
    cv_rmse_scores = np.sqrt(-cv_scores)

    rf_model.fit(X_train_scenario_scaled_selected, y_train_scenario)
    y_pred_scenario = rf_model.predict(
        X_test_scaled[:, [independent_vars.index(f) for f in important_features]]
    )

    mae = mean_absolute_error(y_test, y_pred_scenario)
    r2 = r2_score(y_test, y_pred_scenario)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_scenario))

    results.append(
        {
            "Iteration": i,
            "Images_Used": i,
            "MAE": mae,
            "R2": r2,
            "RMSE": rmse,
            "CV_RMSE_Mean": cv_rmse_scores.mean(),
            "CV_RMSE_Std": cv_rmse_scores.std(),
        }
    )

In [ ]:
# 📊 Final results and plot
results_df = pd.DataFrame(results)
results_df["R2_smooth"] = results_df["R2"].rolling(window=3, center=True).mean()

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="RMSE",
    marker="o",
    label="RMSE (Test)",
    color="blue",
    ax=ax1,
)
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="MAE",
    marker="s",
    label="MAE (Test)",
    color="orange",
    ax=ax1,
)
ax1.set_xlabel("Number of Images Used")
ax1.set_ylabel("RMSE & MAE")
ax1.legend(loc="upper left")

In [ ]:
ax2 = ax1.twinx()
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="R2_smooth",
    marker="^",
    label="R² (Smoothed)",
    color="red",
    ax=ax2,
)
ax2.set_ylabel("R² Score")
ax2.legend(loc="upper right")

In [ ]:
plt.title("Performance Metrics vs. Number of Images Used (After Removing Drop Images)")
plt.grid(True)
plt.tight_layout()
plt.show()

# **کد پیشرفته**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LassoCV
from sklearn.metrics import (
    make_scorer,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler

In [ ]:
# 1️⃣ Load data
data = pd.read_csv("/content/TrainingSession_Etezadi.csv").dropna()
print(data.shape)

In [ ]:
# 2️⃣ Convert soilTypes column to binary features
data["soilTypes"] = (
    data["soilTypes"].str.strip("[]").str.replace("'", "").str.split(", ")
)
vectorizer = CountVectorizer(tokenizer=lambda x: x, lowercase=False, preprocessor=None)
soil_features = vectorizer.fit_transform(data["soilTypes"]).toarray()
soil_feature_names = vectorizer.get_feature_names_out()
soil_features_df = pd.DataFrame(soil_features, columns=soil_feature_names)
data = pd.concat([data.reset_index(drop=True), soil_features_df], axis=1)

In [ ]:
# 3️⃣ Define target and independent variables
soil_vars = list(soil_features_df.columns)
independent_vars = [
    "BI_mean",
    "CI_mean",
    "NDMI_mean",
    "OMI_mean",
    "RI_mean",
    "SI_mean",
    "EVI_mean",
    "SAVI_mean",
    "NDVI_mean",
    "BSI_mean",
    "CAI_mean",
    "BI_stdDev",
    "CI_stdDev",
    "NDMI_stdDev",
    "OMI_stdDev",
    "RI_stdDev",
    "SI_stdDev",
    "EVI_stdDev",
    "SAVI_stdDev",
    "NDVI_stdDev",
    "BSI_stdDev",
    "CAI_stdDev",
] + soil_vars

In [ ]:
y = np.log10(data["mean_Om_p"])
X = data[independent_vars]

In [ ]:
# 4️⃣ Train-test split based on Field_no
unique_fields = data["Field_no"].unique()
np.random.seed(42)
test_fields = np.random.choice(
    unique_fields, size=int(0.2 * len(unique_fields)), replace=False
)
train_fields = np.setdiff1d(unique_fields, test_fields)

In [ ]:
train_data = data[data["Field_no"].isin(train_fields)]
test_data = data[data["Field_no"].isin(test_fields)]

In [ ]:
X_train = train_data[independent_vars]
y_train = np.log10(train_data["mean_Om_p"])
X_test = test_data[independent_vars]
y_test = np.log10(test_data["mean_Om_p"])

In [ ]:
# 5️⃣ Standard scaling (only once)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# 6️⃣ Feature Selection with LassoCV
lasso_cv = LassoCV(cv=5, random_state=42, max_iter=100000)
lasso_cv.fit(X_train_scaled, y_train)

In [ ]:
# Extract selected features
lasso_coefficients = pd.Series(lasso_cv.coef_, index=independent_vars)
important_features = lasso_coefficients[lasso_coefficients != 0].index.tolist()
print("Selected Features:", important_features)

In [ ]:
# 7️⃣ Rank images using Lasso CV-RMSE per image
image_rmse_records = []
kf = KFold(n_splits=3, shuffle=True, random_state=42)
rmse_scorer = make_scorer(mean_squared_error, squared=False)

In [ ]:
for img_id, df_img in train_data.groupby("Image_id"):
    if len(df_img) < 10:
        continue

    X_img = df_img[independent_vars]
    y_img = np.log10(df_img["mean_Om_p"])

    scaler_img = StandardScaler()
    X_img_scaled = scaler_img.fit_transform(X_img)

    lasso_model = LassoCV(cv=3, random_state=42, max_iter=10000)
    scores = cross_val_score(
        lasso_model, X_img_scaled, y_img, cv=kf, scoring=rmse_scorer
    )
    rmse_cv_mean = np.mean(scores)

    image_rmse_records.append({"Image_id": img_id, "RMSE": rmse_cv_mean})

In [ ]:
ranked_images_train = (
    pd.DataFrame(image_rmse_records).sort_values(by="RMSE").reset_index(drop=True)
)

In [ ]:
# 8️⃣ Progressive training with RF model
results = []
for i in range(1, len(ranked_images_train) + 1):
    selected_image_ids = ranked_images_train.iloc[:i]["Image_id"].tolist()
    scenario_data = train_data[train_data["Image_id"].isin(selected_image_ids)]

    X_train_scenario_raw = scenario_data[independent_vars]
    y_train_scenario = np.log10(scenario_data["mean_Om_p"])

    X_train_scenario_scaled = scaler.transform(X_train_scenario_raw)
    X_train_scenario_scaled_selected = pd.DataFrame(
        X_train_scenario_scaled, columns=independent_vars
    )[important_features].values

    rf_model = RandomForestRegressor(random_state=42)
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(
        rf_model,
        X_train_scenario_scaled_selected,
        y_train_scenario,
        cv=kfold,
        scoring="neg_mean_squared_error",
    )
    cv_rmse_scores = np.sqrt(-cv_scores)

    rf_model.fit(X_train_scenario_scaled_selected, y_train_scenario)
    y_pred_scenario = rf_model.predict(
        X_test_scaled[:, [independent_vars.index(f) for f in important_features]]
    )

    mae = mean_absolute_error(y_test, y_pred_scenario)
    r2 = r2_score(y_test, y_pred_scenario)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_scenario))

    results.append(
        {
            "Iteration": i,
            "Images_Used": i,
            "MAE": mae,
            "R2": r2,
            "RMSE": rmse,
            "CV_RMSE_Mean": cv_rmse_scores.mean(),
            "CV_RMSE_Std": cv_rmse_scores.std(),
        }
    )

In [ ]:
results_df = pd.DataFrame(results)

In [ ]:
# 🔁 Identify and remove bad images based on R2 drop threshold
drop_threshold = 0.01
bad_iterations = []
for i in range(1, len(results_df)):
    if results_df.loc[i, "R2"] < results_df.loc[i - 1, "R2"] - drop_threshold:
        bad_iterations.append(i)

In [ ]:
bad_images = ranked_images_train.iloc[bad_iterations]["Image_id"].tolist()
print("Bad images detected:", bad_images)

In [ ]:
# Remove bad images from ranking
ranked_images_train = ranked_images_train[
    ~ranked_images_train["Image_id"].isin(bad_images)
].reset_index(drop=True)

In [ ]:
# 🔁 Re-run RF training on good images only
results = []
for i in range(1, len(ranked_images_train) + 1):
    selected_image_ids = ranked_images_train.iloc[:i]["Image_id"].tolist()
    scenario_data = train_data[train_data["Image_id"].isin(selected_image_ids)]

    X_train_scenario_raw = scenario_data[independent_vars]
    y_train_scenario = np.log10(scenario_data["mean_Om_p"])

    X_train_scenario_scaled = scaler.transform(X_train_scenario_raw)
    X_train_scenario_scaled_selected = pd.DataFrame(
        X_train_scenario_scaled, columns=independent_vars
    )[important_features].values

    rf_model = RandomForestRegressor(random_state=42)
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(
        rf_model,
        X_train_scenario_scaled_selected,
        y_train_scenario,
        cv=kfold,
        scoring="neg_mean_squared_error",
    )
    cv_rmse_scores = np.sqrt(-cv_scores)

    rf_model.fit(X_train_scenario_scaled_selected, y_train_scenario)
    y_pred_scenario = rf_model.predict(
        X_test_scaled[:, [independent_vars.index(f) for f in important_features]]
    )

    mae = mean_absolute_error(y_test, y_pred_scenario)
    r2 = r2_score(y_test, y_pred_scenario)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_scenario))

    results.append(
        {
            "Iteration": i,
            "Images_Used": i,
            "MAE": mae,
            "R2": r2,
            "RMSE": rmse,
            "CV_RMSE_Mean": cv_rmse_scores.mean(),
            "CV_RMSE_Std": cv_rmse_scores.std(),
        }
    )

In [ ]:
# 📊 Final results and plot
results_df = pd.DataFrame(results)
results_df["R2_smooth"] = results_df["R2"].rolling(window=3, center=True).mean()
results_df["R2_ewm"] = results_df["R2"].ewm(span=3, adjust=False).mean()

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="RMSE",
    marker="o",
    label="RMSE (Test)",
    color="blue",
    ax=ax1,
)
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="MAE",
    marker="s",
    label="MAE (Test)",
    color="orange",
    ax=ax1,
)
ax1.set_xlabel("Number of Images Used")
ax1.set_ylabel("RMSE & MAE")
ax1.legend(loc="upper left")

In [ ]:
ax2 = ax1.twinx()
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="R2_smooth",
    marker="^",
    label="R² (Smoothed)",
    color="red",
    ax=ax2,
)
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="R2_ewm",
    marker="x",
    label="R² (EWMA)",
    color="green",
    ax=ax2,
)
ax2.set_ylabel("R² Score")
ax2.legend(loc="upper right")

In [ ]:
plt.title("Performance Metrics vs. Number of Images Used (After Removing Drop Images)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.signal import savgol_filter
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LassoCV
from sklearn.metrics import (
    make_scorer,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler

In [ ]:
# 1️⃣ Load data
data = pd.read_csv("/content/TrainingSession_Etezadi.csv").dropna()
print(data.shape)

In [ ]:
# 2️⃣ Convert soilTypes column to binary features
data["soilTypes"] = (
    data["soilTypes"].str.strip("[]").str.replace("'", "").str.split(", ")
)
vectorizer = CountVectorizer(tokenizer=lambda x: x, lowercase=False, preprocessor=None)
soil_features = vectorizer.fit_transform(data["soilTypes"]).toarray()
soil_feature_names = vectorizer.get_feature_names_out()
soil_features_df = pd.DataFrame(soil_features, columns=soil_feature_names)
data = pd.concat([data.reset_index(drop=True), soil_features_df], axis=1)

In [ ]:
# 3️⃣ Define target and independent variables
soil_vars = list(soil_features_df.columns)
independent_vars = [
    "BI_mean",
    "CI_mean",
    "NDMI_mean",
    "OMI_mean",
    "RI_mean",
    "SI_mean",
    "EVI_mean",
    "SAVI_mean",
    "NDVI_mean",
    "BSI_mean",
    "CAI_mean",
    "BI_stdDev",
    "CI_stdDev",
    "NDMI_stdDev",
    "OMI_stdDev",
    "RI_stdDev",
    "SI_stdDev",
    "EVI_stdDev",
    "SAVI_stdDev",
    "NDVI_stdDev",
    "BSI_stdDev",
    "CAI_stdDev",
] + soil_vars

In [ ]:
y = np.log10(data["mean_Om_p"])
X = data[independent_vars]

In [ ]:
# 4️⃣ Train-test split based on Field_no
unique_fields = data["Field_no"].unique()
np.random.seed(42)
test_fields = np.random.choice(
    unique_fields, size=int(0.2 * len(unique_fields)), replace=False
)
train_fields = np.setdiff1d(unique_fields, test_fields)

In [ ]:
train_data = data[data["Field_no"].isin(train_fields)]
test_data = data[data["Field_no"].isin(test_fields)]

In [ ]:
X_train = train_data[independent_vars]
y_train = np.log10(train_data["mean_Om_p"])
X_test = test_data[independent_vars]
y_test = np.log10(test_data["mean_Om_p"])

In [ ]:
# 5️⃣ Standard scaling (only once)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# 6️⃣ Feature Selection with LassoCV
lasso_cv = LassoCV(cv=5, random_state=42, max_iter=100000)
lasso_cv.fit(X_train_scaled, y_train)

In [ ]:
# Extract selected features
lasso_coefficients = pd.Series(lasso_cv.coef_, index=independent_vars)
important_features = lasso_coefficients[lasso_coefficients != 0].index.tolist()
print("Selected Features:", important_features)

In [ ]:
# 7️⃣ Rank images using Lasso CV-RMSE per image
image_rmse_records = []
kf = KFold(n_splits=3, shuffle=True, random_state=42)
rmse_scorer = make_scorer(mean_squared_error, squared=False)

In [ ]:
for img_id, df_img in train_data.groupby("Image_id"):
    if len(df_img) < 10:
        continue

    X_img = df_img[independent_vars]
    y_img = np.log10(df_img["mean_Om_p"])

    scaler_img = StandardScaler()
    X_img_scaled = scaler_img.fit_transform(X_img)

    lasso_model = LassoCV(cv=3, random_state=42, max_iter=10000)
    scores = cross_val_score(
        lasso_model, X_img_scaled, y_img, cv=kf, scoring=rmse_scorer
    )
    rmse_cv_mean = np.mean(scores)

    image_rmse_records.append({"Image_id": img_id, "RMSE": rmse_cv_mean})

In [ ]:
ranked_images_train = (
    pd.DataFrame(image_rmse_records).sort_values(by="RMSE").reset_index(drop=True)
)

In [ ]:
# 8️⃣ Progressive training with RF model
results = []
for i in range(1, len(ranked_images_train) + 1):
    selected_image_ids = ranked_images_train.iloc[:i]["Image_id"].tolist()
    scenario_data = train_data[train_data["Image_id"].isin(selected_image_ids)]

    X_train_scenario_raw = scenario_data[independent_vars]
    y_train_scenario = np.log10(scenario_data["mean_Om_p"])

    X_train_scenario_scaled = scaler.transform(X_train_scenario_raw)
    X_train_scenario_scaled_selected = pd.DataFrame(
        X_train_scenario_scaled, columns=independent_vars
    )[important_features].values

    rf_model = RandomForestRegressor(random_state=42)
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(
        rf_model,
        X_train_scenario_scaled_selected,
        y_train_scenario,
        cv=kfold,
        scoring="neg_mean_squared_error",
    )
    cv_rmse_scores = np.sqrt(-cv_scores)

    rf_model.fit(X_train_scenario_scaled_selected, y_train_scenario)
    y_pred_scenario = rf_model.predict(
        X_test_scaled[:, [independent_vars.index(f) for f in important_features]]
    )

    mae = mean_absolute_error(y_test, y_pred_scenario)
    r2 = r2_score(y_test, y_pred_scenario)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_scenario))

    results.append(
        {
            "Iteration": i,
            "Images_Used": i,
            "MAE": mae,
            "R2": r2,
            "RMSE": rmse,
            "CV_RMSE_Mean": cv_rmse_scores.mean(),
            "CV_RMSE_Std": cv_rmse_scores.std(),
        }
    )

In [ ]:
results_df = pd.DataFrame(results)

In [ ]:
# 🔁 Identify and remove bad images based on R2 drop threshold
drop_threshold = 0.01
bad_iterations = []
for i in range(1, len(results_df)):
    if results_df.loc[i, "R2"] < results_df.loc[i - 1, "R2"] - drop_threshold:
        bad_iterations.append(i)

In [ ]:
bad_images = ranked_images_train.iloc[bad_iterations]["Image_id"].tolist()
print("Bad images detected:", bad_images)

In [ ]:
# Remove bad images from ranking
ranked_images_train = ranked_images_train[
    ~ranked_images_train["Image_id"].isin(bad_images)
].reset_index(drop=True)

In [ ]:
# 🔁 Re-run RF training on good images only
results = []
for i in range(1, len(ranked_images_train) + 1):
    selected_image_ids = ranked_images_train.iloc[:i]["Image_id"].tolist()
    scenario_data = train_data[train_data["Image_id"].isin(selected_image_ids)]

    X_train_scenario_raw = scenario_data[independent_vars]
    y_train_scenario = np.log10(scenario_data["mean_Om_p"])

    X_train_scenario_scaled = scaler.transform(X_train_scenario_raw)
    X_train_scenario_scaled_selected = pd.DataFrame(
        X_train_scenario_scaled, columns=independent_vars
    )[important_features].values

    rf_model = RandomForestRegressor(random_state=42)
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(
        rf_model,
        X_train_scenario_scaled_selected,
        y_train_scenario,
        cv=kfold,
        scoring="neg_mean_squared_error",
    )
    cv_rmse_scores = np.sqrt(-cv_scores)

    rf_model.fit(X_train_scenario_scaled_selected, y_train_scenario)
    y_pred_scenario = rf_model.predict(
        X_test_scaled[:, [independent_vars.index(f) for f in important_features]]
    )

    mae = mean_absolute_error(y_test, y_pred_scenario)
    r2 = r2_score(y_test, y_pred_scenario)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_scenario))

    results.append(
        {
            "Iteration": i,
            "Images_Used": i,
            "MAE": mae,
            "R2": r2,
            "RMSE": rmse,
            "CV_RMSE_Mean": cv_rmse_scores.mean(),
            "CV_RMSE_Std": cv_rmse_scores.std(),
        }
    )

In [ ]:
# 📊 Final results and plot
results_df = pd.DataFrame(results)
results_df["R2_smooth"] = results_df["R2"].rolling(window=3, center=True).mean()
results_df["R2_ewm"] = results_df["R2"].ewm(span=3, adjust=False).mean()
results_df["R2_savgol"] = savgol_filter(
    results_df["R2"],
    window_length=(
        5 if len(results_df) >= 5 else len(results_df) - (len(results_df) % 2 == 0)
    ),
    polyorder=2,
)

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="RMSE",
    marker="o",
    label="RMSE (Test)",
    color="blue",
    ax=ax1,
)
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="MAE",
    marker="s",
    label="MAE (Test)",
    color="orange",
    ax=ax1,
)
ax1.set_xlabel("Number of Images Used")
ax1.set_ylabel("RMSE & MAE")
ax1.legend(loc="upper left")

In [ ]:
ax2 = ax1.twinx()
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="R2_smooth",
    marker="^",
    label="R² (Rolling)",
    color="red",
    ax=ax2,
)
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="R2_ewm",
    marker="x",
    label="R² (EWMA)",
    color="green",
    ax=ax2,
)
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="R2_savgol",
    marker="D",
    label="R² (Savitzky-Golay)",
    color="purple",
    ax=ax2,
)
ax2.set_ylabel("R² Score")
ax2.legend(loc="upper right")

In [ ]:
plt.title("Performance Metrics vs. Number of Images Used (After Removing Drop Images)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.signal import savgol_filter
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LassoCV
from sklearn.metrics import (
    make_scorer,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler

In [ ]:
# 1️⃣ Load data
data = pd.read_csv("/content/TrainingSession_Etezadi.csv").dropna()
print(data.shape)

In [ ]:
# 2️⃣ Convert soilTypes column to binary features
data["soilTypes"] = (
    data["soilTypes"].str.strip("[]").str.replace("'", "").str.split(", ")
)
vectorizer = CountVectorizer(tokenizer=lambda x: x, lowercase=False, preprocessor=None)
soil_features = vectorizer.fit_transform(data["soilTypes"]).toarray()
soil_feature_names = vectorizer.get_feature_names_out()
soil_features_df = pd.DataFrame(soil_features, columns=soil_feature_names)
data = pd.concat([data.reset_index(drop=True), soil_features_df], axis=1)

In [ ]:
# 3️⃣ Define target and independent variables
soil_vars = list(soil_features_df.columns)
independent_vars = [
    "BI_mean",
    "CI_mean",
    "NDMI_mean",
    "OMI_mean",
    "RI_mean",
    "SI_mean",
    "EVI_mean",
    "SAVI_mean",
    "NDVI_mean",
    "BSI_mean",
    "CAI_mean",
    "BI_stdDev",
    "CI_stdDev",
    "NDMI_stdDev",
    "OMI_stdDev",
    "RI_stdDev",
    "SI_stdDev",
    "EVI_stdDev",
    "SAVI_stdDev",
    "NDVI_stdDev",
    "BSI_stdDev",
    "CAI_stdDev",
] + soil_vars

In [ ]:
y = np.log10(data["mean_Om_p"])
X = data[independent_vars]

In [ ]:
# 4️⃣ Train-test split based on Field_no
unique_fields = data["Field_no"].unique()
np.random.seed(42)
test_fields = np.random.choice(
    unique_fields, size=int(0.2 * len(unique_fields)), replace=False
)
train_fields = np.setdiff1d(unique_fields, test_fields)

In [ ]:
train_data = data[data["Field_no"].isin(train_fields)]
test_data = data[data["Field_no"].isin(test_fields)]

In [ ]:
X_test = test_data[independent_vars]
y_test = np.log10(test_data["mean_Om_p"])

In [ ]:
# 5️⃣ Feature Selection with LassoCV
scaler_full = StandardScaler()
X_train_scaled_full = scaler_full.fit_transform(train_data[independent_vars])
y_train_full = np.log10(train_data["mean_Om_p"])

In [ ]:
lasso_cv = LassoCV(cv=5, random_state=42, max_iter=100000)
lasso_cv.fit(X_train_scaled_full, y_train_full)

In [ ]:
lasso_coefficients = pd.Series(lasso_cv.coef_, index=independent_vars)
important_features = lasso_coefficients[lasso_coefficients != 0].index.tolist()
print("Selected Features:", important_features)

In [ ]:
# 6️⃣ Rank images using Lasso CV-RMSE per image
image_rmse_records = []
kf = KFold(n_splits=3, shuffle=True, random_state=42)
rmse_scorer = make_scorer(mean_squared_error, squared=False)

In [ ]:
for img_id, df_img in train_data.groupby("Image_id"):
    if len(df_img) < 10:
        continue

    X_img = df_img[independent_vars]
    y_img = np.log10(df_img["mean_Om_p"])

    scaler_img = StandardScaler()
    X_img_scaled = scaler_img.fit_transform(X_img)

    lasso_model = LassoCV(cv=3, random_state=42, max_iter=10000)
    scores = cross_val_score(
        lasso_model, X_img_scaled, y_img, cv=kf, scoring=rmse_scorer
    )
    rmse_cv_mean = np.mean(scores)

    image_rmse_records.append({"Image_id": img_id, "RMSE": rmse_cv_mean})

In [ ]:
ranked_images_train = (
    pd.DataFrame(image_rmse_records).sort_values(by="RMSE").reset_index(drop=True)
)

In [ ]:
# 🔍 Identify and remove bad images with R² drop
r2_drop_threshold = 0.01
bad_iterations = []

In [ ]:
# Dummy run to get R2 values
r2_track = []
for i in range(1, len(ranked_images_train) + 1):
    selected_image_ids = ranked_images_train.iloc[:i]["Image_id"].tolist()
    scenario_data = train_data[train_data["Image_id"].isin(selected_image_ids)]
    X_train_scenario_raw = scenario_data[independent_vars]
    y_train_scenario = np.log10(scenario_data["mean_Om_p"])
    scaler_scenario = StandardScaler()
    X_train_scenario_scaled = scaler_scenario.fit_transform(X_train_scenario_raw)
    X_train_scenario_scaled_selected = pd.DataFrame(
        X_train_scenario_scaled, columns=independent_vars
    )[important_features].values
    X_test_scaled = scaler_scenario.transform(X_test)
    X_test_scaled_selected = pd.DataFrame(X_test_scaled, columns=independent_vars)[
        important_features
    ].values
    rf_model = RandomForestRegressor(random_state=42)
    rf_model.fit(X_train_scenario_scaled_selected, y_train_scenario)
    y_pred = rf_model.predict(X_test_scaled_selected)
    r2 = r2_score(y_test, y_pred)
    r2_track.append(r2)
    if i > 1 and r2 < r2_track[i - 2] - r2_drop_threshold:
        bad_iterations.append(i - 1)

In [ ]:
bad_images = ranked_images_train.iloc[bad_iterations]["Image_id"].tolist()
ranked_images_train = ranked_images_train[
    ~ranked_images_train["Image_id"].isin(bad_images)
].reset_index(drop=True)

In [ ]:
# 7️⃣ Progressive training with RF model (cleaned)
results = []
for i in range(1, len(ranked_images_train) + 1):
    selected_image_ids = ranked_images_train.iloc[:i]["Image_id"].tolist()
    scenario_data = train_data[train_data["Image_id"].isin(selected_image_ids)]

    X_train_scenario_raw = scenario_data[independent_vars]
    y_train_scenario = np.log10(scenario_data["mean_Om_p"])

    scaler_scenario = StandardScaler()
    X_train_scenario_scaled = scaler_scenario.fit_transform(X_train_scenario_raw)
    X_train_scenario_scaled_selected = pd.DataFrame(
        X_train_scenario_scaled, columns=independent_vars
    )[important_features].values

    X_test_scenario_scaled = scaler_scenario.transform(X_test)
    X_test_scenario_scaled_selected = pd.DataFrame(
        X_test_scenario_scaled, columns=independent_vars
    )[important_features].values

    rf_model = RandomForestRegressor(random_state=42)
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(
        rf_model,
        X_train_scenario_scaled_selected,
        y_train_scenario,
        cv=kfold,
        scoring="neg_mean_squared_error",
    )
    cv_rmse_scores = np.sqrt(-cv_scores)

    rf_model.fit(X_train_scenario_scaled_selected, y_train_scenario)
    y_pred_scenario = rf_model.predict(X_test_scenario_scaled_selected)

    mae = mean_absolute_error(y_test, y_pred_scenario)
    r2 = r2_score(y_test, y_pred_scenario)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_scenario))

    results.append(
        {
            "Iteration": i,
            "Images_Used": i,
            "MAE": mae,
            "R2": r2,
            "RMSE": rmse,
            "CV_RMSE_Mean": cv_rmse_scores.mean(),
            "CV_RMSE_Std": cv_rmse_scores.std(),
        }
    )

In [ ]:
results_df = pd.DataFrame(results)

In [ ]:
# 🔁 Smoothing R2 using multiple methods
results_df["R2_smooth"] = results_df["R2"].rolling(window=3, center=True).mean()
results_df["R2_ewm"] = results_df["R2"].ewm(span=3, adjust=False).mean()
results_df["R2_savgol"] = savgol_filter(
    results_df["R2"],
    window_length=(
        5 if len(results_df) >= 5 else len(results_df) - (len(results_df) % 2 == 0)
    ),
    polyorder=2,
)

In [ ]:
# 📊 Final Plot
fig, ax1 = plt.subplots(figsize=(12, 6))
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="RMSE",
    marker="o",
    label="RMSE (Test)",
    color="blue",
    ax=ax1,
)
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="MAE",
    marker="s",
    label="MAE (Test)",
    color="orange",
    ax=ax1,
)
ax1.set_xlabel("Number of Images Used")
ax1.set_ylabel("RMSE & MAE")
ax1.legend(loc="upper left")

In [ ]:
ax2 = ax1.twinx()
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="R2_smooth",
    marker="^",
    label="R² (Rolling)",
    color="red",
    ax=ax2,
)
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="R2_ewm",
    marker="x",
    label="R² (EWMA)",
    color="green",
    ax=ax2,
)
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="R2_savgol",
    marker="D",
    label="R² (Savitzky-Golay)",
    color="purple",
    ax=ax2,
)
ax2.set_ylabel("R² Score")
ax2.legend(loc="upper right")

In [ ]:
plt.title("Performance Metrics vs. Number of Images Used (After Removing Bad Images)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.signal import savgol_filter
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LassoCV
from sklearn.metrics import (
    make_scorer,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler

In [ ]:
# 1️⃣ Load data
data = pd.read_csv("/content/TrainingSession_Etezadi.csv").dropna()
print(data.shape)

In [ ]:
# 2️⃣ Convert soilTypes column to binary features
data["soilTypes"] = (
    data["soilTypes"].str.strip("[]").str.replace("'", "").str.split(", ")
)
vectorizer = CountVectorizer(tokenizer=lambda x: x, lowercase=False, preprocessor=None)
soil_features = vectorizer.fit_transform(data["soilTypes"]).toarray()
soil_feature_names = vectorizer.get_feature_names_out()
soil_features_df = pd.DataFrame(soil_features, columns=soil_feature_names)
data = pd.concat([data.reset_index(drop=True), soil_features_df], axis=1)

In [ ]:
# 3️⃣ Define target and independent variables
soil_vars = list(soil_features_df.columns)
independent_vars = [
    "BI_mean",
    "CI_mean",
    "NDMI_mean",
    "OMI_mean",
    "RI_mean",
    "SI_mean",
    "EVI_mean",
    "SAVI_mean",
    "NDVI_mean",
    "BSI_mean",
    "CAI_mean",
    "BI_stdDev",
    "CI_stdDev",
    "NDMI_stdDev",
    "OMI_stdDev",
    "RI_stdDev",
    "SI_stdDev",
    "EVI_stdDev",
    "SAVI_stdDev",
    "NDVI_stdDev",
    "BSI_stdDev",
    "CAI_stdDev",
] + soil_vars

In [ ]:
y = np.log10(data["mean_Om_p"])
X = data[independent_vars]

In [ ]:
# 4️⃣ Train-test split based on Field_no
unique_fields = data["Field_no"].unique()
np.random.seed(42)
test_fields = np.random.choice(
    unique_fields, size=int(0.2 * len(unique_fields)), replace=False
)
train_fields = np.setdiff1d(unique_fields, test_fields)

In [ ]:
train_data = data[data["Field_no"].isin(train_fields)]
test_data = data[data["Field_no"].isin(test_fields)]

In [ ]:
X_test = test_data[independent_vars]
y_test = np.log10(test_data["mean_Om_p"])

In [ ]:
# 5️⃣ Feature Selection with LassoCV
scaler_full = StandardScaler()
X_train_scaled_full = scaler_full.fit_transform(train_data[independent_vars])
y_train_full = np.log10(train_data["mean_Om_p"])

In [ ]:
lasso_cv = LassoCV(cv=5, random_state=42, max_iter=100000)
lasso_cv.fit(X_train_scaled_full, y_train_full)

In [ ]:
lasso_coefficients = pd.Series(lasso_cv.coef_, index=independent_vars)
important_features = lasso_coefficients[lasso_coefficients != 0].index.tolist()
print("Selected Features:", important_features)

In [ ]:
# 6️⃣ Rank images using Lasso CV-RMSE per image
image_rmse_records = []
kf = KFold(n_splits=3, shuffle=True, random_state=42)
rmse_scorer = make_scorer(mean_squared_error, squared=False)

In [ ]:
for img_id, df_img in train_data.groupby("Image_id"):
    if len(df_img) < 10:
        continue

    X_img = df_img[independent_vars]
    y_img = np.log10(df_img["mean_Om_p"])

    scaler_img = StandardScaler()
    X_img_scaled = scaler_img.fit_transform(X_img)

    lasso_model = LassoCV(cv=3, random_state=42, max_iter=10000)
    scores = cross_val_score(
        lasso_model, X_img_scaled, y_img, cv=kf, scoring=rmse_scorer
    )
    rmse_cv_mean = np.mean(scores)

    image_rmse_records.append({"Image_id": img_id, "RMSE": rmse_cv_mean})

In [ ]:
ranked_images_train = (
    pd.DataFrame(image_rmse_records).sort_values(by="RMSE").reset_index(drop=True)
)

In [ ]:
# 🔍 Automatically find best threshold to remove bad images
def auto_threshold_selection(r2_track, max_threshold=0.05, step=0.001):
    best_threshold = 0
    best_r2 = -np.inf
    thresholds = np.arange(0.001, max_threshold, step)
    for thresh in thresholds:
        r2_copy = r2_track.copy()
        drop_indices = [
            i - 1
            for i in range(2, len(r2_copy))
            if r2_copy[i] < r2_copy[i - 2] - thresh
        ]
        r2_masked = [r2_copy[i] for i in range(len(r2_copy)) if i not in drop_indices]
        avg_r2 = np.mean(r2_masked) if r2_masked else -np.inf
        if avg_r2 > best_r2:
            best_r2 = avg_r2
            best_threshold = thresh
    return best_threshold

In [ ]:
# Dummy run to get R2 values
r2_track = []
for i in range(1, len(ranked_images_train) + 1):
    selected_image_ids = ranked_images_train.iloc[:i]["Image_id"].tolist()
    scenario_data = train_data[train_data["Image_id"].isin(selected_image_ids)]
    X_train_scenario_raw = scenario_data[independent_vars]
    y_train_scenario = np.log10(scenario_data["mean_Om_p"])
    scaler_scenario = StandardScaler()
    X_train_scenario_scaled = scaler_scenario.fit_transform(X_train_scenario_raw)
    X_train_scenario_scaled_selected = pd.DataFrame(
        X_train_scenario_scaled, columns=independent_vars
    )[important_features].values
    X_test_scaled = scaler_scenario.transform(X_test)
    X_test_scaled_selected = pd.DataFrame(X_test_scaled, columns=independent_vars)[
        important_features
    ].values
    rf_model = RandomForestRegressor(random_state=42)
    rf_model.fit(X_train_scenario_scaled_selected, y_train_scenario)
    y_pred = rf_model.predict(X_test_scaled_selected)
    r2 = r2_score(y_test, y_pred)
    r2_track.append(r2)

In [ ]:
# Select best threshold and remove bad images
r2_drop_threshold = auto_threshold_selection(r2_track)
bad_iterations = [
    i - 1
    for i in range(2, len(r2_track))
    if r2_track[i] < r2_track[i - 2] - r2_drop_threshold
]
bad_images = ranked_images_train.iloc[bad_iterations]["Image_id"].tolist()
ranked_images_train = ranked_images_train[
    ~ranked_images_train["Image_id"].isin(bad_images)
].reset_index(drop=True)

In [ ]:
# 7️⃣ Progressive training with RF model (cleaned)
results = []
for i in range(1, len(ranked_images_train) + 1):
    selected_image_ids = ranked_images_train.iloc[:i]["Image_id"].tolist()
    scenario_data = train_data[train_data["Image_id"].isin(selected_image_ids)]

    X_train_scenario_raw = scenario_data[independent_vars]
    y_train_scenario = np.log10(scenario_data["mean_Om_p"])

    scaler_scenario = StandardScaler()
    X_train_scenario_scaled = scaler_scenario.fit_transform(X_train_scenario_raw)
    X_train_scenario_scaled_selected = pd.DataFrame(
        X_train_scenario_scaled, columns=independent_vars
    )[important_features].values

    X_test_scenario_scaled = scaler_scenario.transform(X_test)
    X_test_scenario_scaled_selected = pd.DataFrame(
        X_test_scenario_scaled, columns=independent_vars
    )[important_features].values

    rf_model = RandomForestRegressor(random_state=42)
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(
        rf_model,
        X_train_scenario_scaled_selected,
        y_train_scenario,
        cv=kfold,
        scoring="neg_mean_squared_error",
    )
    cv_rmse_scores = np.sqrt(-cv_scores)

    rf_model.fit(X_train_scenario_scaled_selected, y_train_scenario)
    y_pred_scenario = rf_model.predict(X_test_scenario_scaled_selected)

    mae = mean_absolute_error(y_test, y_pred_scenario)
    r2 = r2_score(y_test, y_pred_scenario)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_scenario))

    results.append(
        {
            "Iteration": i,
            "Images_Used": i,
            "MAE": mae,
            "R2": r2,
            "RMSE": rmse,
            "CV_RMSE_Mean": cv_rmse_scores.mean(),
            "CV_RMSE_Std": cv_rmse_scores.std(),
        }
    )

In [ ]:
results_df = pd.DataFrame(results)

In [ ]:
# 🔁 Smoothing R2 using multiple methods
results_df["R2_smooth"] = results_df["R2"].rolling(window=3, center=True).mean()
results_df["R2_ewm"] = results_df["R2"].ewm(span=3, adjust=False).mean()
results_df["R2_savgol"] = savgol_filter(
    results_df["R2"],
    window_length=(
        5 if len(results_df) >= 5 else len(results_df) - (len(results_df) % 2 == 0)
    ),
    polyorder=2,
)

In [ ]:
# 📊 Final Plot
fig, ax1 = plt.subplots(figsize=(12, 6))
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="RMSE",
    marker="o",
    label="RMSE (Test)",
    color="blue",
    ax=ax1,
)
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="MAE",
    marker="s",
    label="MAE (Test)",
    color="orange",
    ax=ax1,
)
ax1.set_xlabel("Number of Images Used")
ax1.set_ylabel("RMSE & MAE")
ax1.legend(loc="upper left")

In [ ]:
ax2 = ax1.twinx()
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="R2_smooth",
    marker="^",
    label="R² (Rolling)",
    color="red",
    ax=ax2,
)
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="R2_ewm",
    marker="x",
    label="R² (EWMA)",
    color="green",
    ax=ax2,
)
sns.lineplot(
    data=results_df,
    x="Images_Used",
    y="R2_savgol",
    marker="D",
    label="R² (Savitzky-Golay)",
    color="purple",
    ax=ax2,
)
ax2.set_ylabel("R² Score")
ax2.legend(loc="upper right")

In [ ]:
plt.title("Performance Metrics vs. Number of Images Used (After Removing Bad Images)")
plt.grid(True)
plt.tight_layout()
plt.show()

# **Jadid-30Jan2026**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import (
    GroupKFold,
    GroupShuffleSplit,
    KFold,
    cross_val_score,
)
from sklearn.preprocessing import StandardScaler

In [ ]:
# =========================
# 0) Helper
# =========================
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [ ]:
# =========================
# 1) Load
# =========================
data = (
    pd.read_csv("/content/TrainingSession_Etezadi.csv").dropna().reset_index(drop=True)
)
print("Raw shape:", data.shape)

In [ ]:
# =========================
# 2) soilTypes -> binary (one-hot)
# =========================
# robust parsing (handles spaces/quotes)
data["soilTypes"] = (
    data["soilTypes"]
    .astype(str)
    .str.strip("[]")
    .str.replace("'", "", regex=False)
    .str.split(", ")
)

In [ ]:
vectorizer = CountVectorizer(tokenizer=lambda x: x, lowercase=False, preprocessor=None)
soil_features = vectorizer.fit_transform(data["soilTypes"]).toarray()
soil_feature_names = vectorizer.get_feature_names_out()

In [ ]:
soil_df = pd.DataFrame(soil_features, columns=soil_feature_names)
data = pd.concat([data, soil_df], axis=1)

In [ ]:
# =========================
# 3) Features / Target
# =========================
spectral_features = [
    "BI_mean",
    "CI_mean",
    "NDMI_mean",
    "OMI_mean",
    "RI_mean",
    "SI_mean",
    "EVI_mean",
    "SAVI_mean",
    "NDVI_mean",
    "BSI_mean",
    "CAI_mean",
    "BI_stdDev",
    "CI_stdDev",
    "NDMI_stdDev",
    "OMI_stdDev",
    "RI_stdDev",
    "SI_stdDev",
    "EVI_stdDev",
    "SAVI_stdDev",
    "NDVI_stdDev",
    "BSI_stdDev",
    "CAI_stdDev",
]
independent_vars = spectral_features + list(soil_feature_names)

In [ ]:
# Safety: keep only columns that exist
missing = [c for c in independent_vars if c not in data.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

In [ ]:
y_all = np.log10(data["mean_Om_p"].astype(float))
X_all = data[independent_vars].astype(float)

In [ ]:
# groups (field id) for leakage-safe splitting
groups_all = data["Field_no"].values

In [ ]:
# =========================
# 4) Outer split: Train/Test by Field_no
# =========================
np.random.seed(42)
unique_fields = np.unique(groups_all)
test_size_fields = max(1, int(0.2 * len(unique_fields)))
test_fields = np.random.choice(unique_fields, size=test_size_fields, replace=False)

In [ ]:
is_test = data["Field_no"].isin(test_fields).values
train_df = data[~is_test].reset_index(drop=True)
test_df = data[is_test].reset_index(drop=True)

In [ ]:
print("Train rows:", train_df.shape, "Test rows:", test_df.shape)
print(
    "Train fields:",
    train_df["Field_no"].nunique(),
    "Test fields:",
    test_df["Field_no"].nunique(),
)

In [ ]:
X_train_raw = train_df[independent_vars].astype(float)
y_train = np.log10(train_df["mean_Om_p"].astype(float))
g_train = train_df["Field_no"].values

In [ ]:
X_test_raw = test_df[independent_vars].astype(float)
y_test = np.log10(test_df["mean_Om_p"].astype(float))
g_test = test_df["Field_no"].values

In [ ]:
# =========================
# 5) Inner split: Train_inner / Val_inner by Field_no (NO test leakage)
# =========================
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
inner_train_idx, inner_val_idx = next(gss.split(X_train_raw, y_train, groups=g_train))

In [ ]:
train_inner = train_df.iloc[inner_train_idx].reset_index(drop=True)
val_inner = train_df.iloc[inner_val_idx].reset_index(drop=True)

In [ ]:
X_tr_in_raw = train_inner[independent_vars].astype(float)
y_tr_in = np.log10(train_inner["mean_Om_p"].astype(float))
g_tr_in = train_inner["Field_no"].values

In [ ]:
X_val_in_raw = val_inner[independent_vars].astype(float)
y_val_in = np.log10(val_inner["mean_Om_p"].astype(float))
g_val_in = val_inner["Field_no"].values

In [ ]:
print(
    "Inner-train fields:",
    len(np.unique(g_tr_in)),
    "Inner-val fields:",
    len(np.unique(g_val_in)),
)

In [ ]:
# =========================
# 6) Fit scaler ON inner-train only
# =========================
scaler = StandardScaler()
X_tr_in = scaler.fit_transform(X_tr_in_raw)
X_val_in = scaler.transform(X_val_in_raw)
X_train_scaled_full = scaler.transform(X_train_raw)  # later (final train)
X_test_scaled = scaler.transform(X_test_raw)  # only for final report

In [ ]:
# =========================
# 7) Feature Selection (LassoCV) ON inner-train only
# =========================
lasso_cv = LassoCV(cv=5, random_state=42, max_iter=100000)
lasso_cv.fit(X_tr_in, y_tr_in)

In [ ]:
coef = pd.Series(lasso_cv.coef_, index=independent_vars)
important_features = coef[coef != 0].index.tolist()

In [ ]:
print(f"Selected features ({len(important_features)}):")
print(important_features)

In [ ]:
if len(important_features) == 0:
    raise RuntimeError(
        "Lasso selected 0 features. Consider lowering alpha grid / using ElasticNetCV / removing strict preprocessing."
    )

In [ ]:
# Precompute column indices for speed
feat_idx = [independent_vars.index(f) for f in important_features]

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# =========================
# 7-1) Feature Selection (LassoCV) barchart
# =========================
import numpy as np
import pandas as pd

=========================================================
LASSO SELECTED FEATURES PLOTS (add right after Lasso fit)
Assumes you already have:
  - coef: pd.Series(lasso.coef_, index=independent_vars)
  - important_features: list of selected feature names (coef != 0)
  - spectral_features: list of satellite features (your 22 features)
=========================================================

In [ ]:
# --- 0) Safety checks
if "coef" not in globals():
    raise RuntimeError(
        "coef is not defined. Make sure you run: coef = pd.Series(lasso.coef_, index=independent_vars)"
    )
if "important_features" not in globals():
    important_features = coef[coef != 0].index.tolist()

In [ ]:
# sort by absolute coefficient magnitude (for clearer plot)
coef_sorted = coef.reindex(coef.abs().sort_values(ascending=False).index)

In [ ]:
# =========================
# 1) Plot: ALL features (selected highlighted)
# =========================
top_n = min(40, len(coef_sorted))  # show top 40 by |coef| (change if you want)
coef_top = coef_sorted.iloc[:top_n]

In [ ]:
# mark selected vs not
is_selected = coef_top.index.isin(important_features)

In [ ]:
plt.figure(figsize=(12, 7))
ypos = np.arange(len(coef_top))[::-1]

In [ ]:
plt.barh(ypos, coef_top.values[::-1], alpha=0.9)

In [ ]:
plt.yticks(ypos, coef_top.index[::-1])
plt.axvline(0, linewidth=1)

In [ ]:
# add a simple visual tag "(selected)" next to selected features
yticklabels = []
for name in coef_top.index[::-1]:
    if name in important_features:
        yticklabels.append(f"{name}  ✅")
    else:
        yticklabels.append(name)
plt.gca().set_yticklabels(yticklabels)

In [ ]:
plt.title(f"Lasso coefficients (Top {top_n} by |coef|) — selected features marked ✅")
plt.xlabel("Lasso coefficient (on standardized features)")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

In [ ]:
print(f"\nSelected features (count={len(important_features)}):")
print(important_features)

In [ ]:
# =========================
# 2) Plot: ONLY selected SATELLITE features
# =========================
selected_sat = [f for f in important_features if f in spectral_features]

In [ ]:
if len(selected_sat) == 0:
    print("\n⚠️ No satellite spectral features were selected by Lasso.")
else:
    coef_sat = coef[selected_sat].sort_values(key=lambda s: s.abs(), ascending=True)

    plt.figure(figsize=(10, max(4, 0.35 * len(coef_sat))))
    ypos = np.arange(len(coef_sat))

    plt.barh(ypos, coef_sat.values)
    plt.yticks(ypos, coef_sat.index)
    plt.axvline(0, linewidth=1)

    plt.title("Selected satellite features only (Lasso coefficients)")
    plt.xlabel("Lasso coefficient (on standardized features)")
    plt.ylabel("Satellite feature")
    plt.tight_layout()
    plt.show()

    print(f"\nSelected satellite features (count={len(selected_sat)}):")
    print(selected_sat)

In [ ]:
# =========================
# 8) Rank images (Image_id) ON inner-train using GroupKFold(Field_no)
#    Important: NO nested CV; we do LassoCV inside each fold? -> we avoid that.
#    We will score with a SIMPLE Lasso using alpha = lasso_cv.alpha_ (fixed).
# =========================
from sklearn.linear_model import Lasso

In [ ]:
alpha_fixed = lasso_cv.alpha_
lasso_fixed = Lasso(alpha=alpha_fixed, max_iter=100000, random_state=42)

In [ ]:
image_scores = []
gkf_rank = GroupKFold(n_splits=3)

In [ ]:
# We'll rank based on CV-RMSE per image, using only important features
for img_id, df_img in train_inner.groupby("Image_id"):
    if df_img.shape[0] < 10:
        continue

    X_img_raw = df_img[independent_vars].astype(float)
    y_img = np.log10(df_img["mean_Om_p"].astype(float))
    g_img = df_img["Field_no"].values

    # scale using global scaler fitted on inner-train
    X_img = scaler.transform(X_img_raw)[:, feat_idx]

    # GroupKFold ensures fields don't leak between folds within this image
    cv_rmse = []
    for tr_i, va_i in gkf_rank.split(X_img, y_img, groups=g_img):
        lasso_fixed.fit(X_img[tr_i], y_img.iloc[tr_i])
        pred = lasso_fixed.predict(X_img[va_i])
        cv_rmse.append(rmse(y_img.iloc[va_i], pred))

    image_scores.append(
        {
            "Image_id": img_id,
            "CV_RMSE": float(np.mean(cv_rmse)),
            "n": int(df_img.shape[0]),
        }
    )

In [ ]:
ranked_images = pd.DataFrame(image_scores).sort_values("CV_RMSE").reset_index(drop=True)
print("Ranked images:", ranked_images.shape)
print(ranked_images.head(10))

In [ ]:
# =========================
# 9) Progressive RF training
#    - Fit RF on inner-train rows that belong to selected images
#    - Evaluate ONLY on inner-val (for image filtering & curves)
# =========================
rf_base = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)

In [ ]:
val_results = []
for i in range(1, len(ranked_images) + 1):
    selected_ids = ranked_images.iloc[:i]["Image_id"].tolist()
    tr_scn = train_inner[train_inner["Image_id"].isin(selected_ids)]

    if tr_scn.shape[0] < 30:
        # too small; skip (optional)
        continue

    X_tr_scn = scaler.transform(tr_scn[independent_vars].astype(float))[:, feat_idx]
    y_tr_scn = np.log10(tr_scn["mean_Om_p"].astype(float))

    # Fit
    rf_base.fit(X_tr_scn, y_tr_scn)

    # Predict on validation (full val set, not restricted to image ids)
    X_val = X_val_in[:, feat_idx]
    pred_val = rf_base.predict(X_val)

    val_results.append(
        {
            "Images_Used": i,
            "R2_val": r2_score(y_val_in, pred_val),
            "RMSE_val": rmse(y_val_in, pred_val),
            "MAE_val": mean_absolute_error(y_val_in, pred_val),
            "TrainRows": int(tr_scn.shape[0]),
        }
    )

In [ ]:
val_df = pd.DataFrame(val_results).reset_index(drop=True)
print(val_df.tail())

In [ ]:
# =========================
# 10) Detect & remove bad images (based on validation R2 drops)
# =========================
drop_threshold = 0.01  # same as your logic

In [ ]:
bad_image_ids = []
# map from Images_Used to image_id added at that step
# (step k adds ranked_images[k-1])
for k in range(1, len(val_df)):
    if val_df.loc[k, "R2_val"] < val_df.loc[k - 1, "R2_val"] - drop_threshold:
        # image added at this step:
        img_added = ranked_images.iloc[val_df.loc[k, "Images_Used"] - 1]["Image_id"]
        bad_image_ids.append(img_added)

In [ ]:
bad_image_ids = list(dict.fromkeys(bad_image_ids))  # unique preserve order
print("Bad images detected (VAL-based):", bad_image_ids)

In [ ]:
ranked_good = ranked_images[~ranked_images["Image_id"].isin(bad_image_ids)].reset_index(
    drop=True
)
print("Good ranked images:", ranked_good.shape)

In [ ]:
# =========================
# 11) Re-run progressive RF on GOOD images (validation curves)
# =========================
val_results_good = []
for i in range(1, len(ranked_good) + 1):
    selected_ids = ranked_good.iloc[:i]["Image_id"].tolist()
    tr_scn = train_inner[train_inner["Image_id"].isin(selected_ids)]
    if tr_scn.shape[0] < 30:
        continue

    X_tr_scn = scaler.transform(tr_scn[independent_vars].astype(float))[:, feat_idx]
    y_tr_scn = np.log10(tr_scn["mean_Om_p"].astype(float))

    rf_base.fit(X_tr_scn, y_tr_scn)
    pred_val = rf_base.predict(X_val_in[:, feat_idx])

    val_results_good.append(
        {
            "Images_Used": i,
            "R2_val": r2_score(y_val_in, pred_val),
            "RMSE_val": rmse(y_val_in, pred_val),
            "MAE_val": mean_absolute_error(y_val_in, pred_val),
            "TrainRows": int(tr_scn.shape[0]),
        }
    )

In [ ]:
val_good_df = pd.DataFrame(val_results_good).reset_index(drop=True)

In [ ]:
# Smooth (optional)
val_good_df["R2_smooth"] = val_good_df["R2_val"].rolling(window=3, center=True).mean()
val_good_df["R2_ewm"] = val_good_df["R2_val"].ewm(span=3, adjust=False).mean()

In [ ]:
# =========================
# 12) Choose best number of images using validation (no test leakage)
# =========================
# Example: maximize R2 on validation
best_i = int(val_good_df.loc[val_good_df["R2_val"].idxmax(), "Images_Used"])
print("Best #images by VAL R2:", best_i)

In [ ]:
selected_final_ids = ranked_good.iloc[:best_i]["Image_id"].tolist()

=========================
13) Train FINAL model on FULL TRAIN (outer train) using selected images
    Then evaluate ONCE on TEST
=========================
NOTE: scaler & features were derived from inner-train (leak-free wrt test). Good.

In [ ]:
train_final = train_df[train_df["Image_id"].isin(selected_final_ids)]
X_train_final = scaler.transform(train_final[independent_vars].astype(float))[
    :, feat_idx
]
y_train_final = np.log10(train_final["mean_Om_p"].astype(float))

In [ ]:
rf_final = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
rf_final.fit(X_train_final, y_train_final)

In [ ]:
# Test prediction (full test)
X_test_final = X_test_scaled[:, feat_idx]
pred_test = rf_final.predict(X_test_final)

In [ ]:
print("\n===== FINAL TEST METRICS (ONE-SHOT) =====")
print("MAE_test :", mean_absolute_error(y_test, pred_test))
print("RMSE_test:", rmse(y_test, pred_test))
print("R2_test  :", r2_score(y_test, pred_test))

In [ ]:
# =========================
# 14) Plots (Validation curves + Test point)
# =========================
plt.figure(figsize=(12, 6))
sns.lineplot(
    data=val_good_df, x="Images_Used", y="RMSE_val", marker="o", label="RMSE (VAL)"
)
sns.lineplot(
    data=val_good_df, x="Images_Used", y="MAE_val", marker="s", label="MAE (VAL)"
)
plt.xlabel("Number of Images Used")
plt.ylabel("Error (log10 scale)")
plt.title("Validation Error vs. #Images (after removing bad images)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(
    data=val_good_df,
    x="Images_Used",
    y="R2_smooth",
    marker="^",
    label="R² smooth (VAL)",
)
sns.lineplot(
    data=val_good_df, x="Images_Used", y="R2_ewm", marker="x", label="R² ewm (VAL)"
)
plt.axvline(best_i, linestyle="--", label=f"Chosen best_i={best_i}")
plt.xlabel("Number of Images Used")
plt.ylabel("R² (validation)")
plt.title("Validation R² vs. #Images (after removing bad images)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
!pip -q install geopandas pyproj shapely fiona

In [ ]:
import numpy as np
import pandas as pd

data: همان دیتافریم اصلی CSV تو (بعد از one-hot)
rf_final: مدل نهایی
scaler: استانداردساز fit شده
independent_vars: لیست فیچرها
feat_idx: ایندکس فیچرهای مهم (important_features)

In [ ]:
X_all_scaled = scaler.transform(data[independent_vars].astype(float))[:, feat_idx]
pred_log = rf_final.predict(X_all_scaled)

In [ ]:
pred_df = data[["Field_no", "Image_id", "mean_Om_p"]].copy()
pred_df["som_pred_log"] = pred_log
pred_df["som_pred"] = 10**pred_log  # برگشت از log10 به مقیاس SOM
# pred_df has: Field_no, Image_id, som_pred (linear), mean_Om_p (optional)
pred_df.to_csv("/content/som_field_image_predictions.csv", index=False)
print("Saved:", "/content/som_field_image_predictions.csv")

In [ ]:
# تجمیع به سطح Field
field_pred = pred_df.groupby("Field_no", as_index=False).agg(
    som_pred=("som_pred", "median"),  # پیشنهاد: median مقاوم‌تر از mean
    som_pred_log=("som_pred_log", "median"),
    som_obs=("mean_Om_p", "mean"),
    n_images=("Image_id", "nunique"),
    n_rows=("Image_id", "size"),
)

In [ ]:
field_pred.to_csv("som_field_predictions.csv", index=False)
field_pred.head()

In [ ]:
import os

In [ ]:
import geopandas as gpd

In [ ]:
os.environ["SHAPE_RESTORE_SHX"] = "YES"
fields_gdf = gpd.read_file("/content/Field_boundary.shp")

In [ ]:
print(fields_gdf.columns)

In [ ]:
import os

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
os.environ["SHAPE_RESTORE_SHX"] = "YES"

In [ ]:
# مسیر shapefile فیلدها
fields_path = "/content/Field_boundary.shp"  # <-- عوض کن (یا .gpkg)
fields_gdf = gpd.read_file(fields_path)

In [ ]:
# جدول پیش‌بینی
pred = pd.read_csv("som_field_predictions.csv")

In [ ]:
# ---- تنظیم کلید join ----
# اگر در shapefile ستون FIELD داری:
# join_left = "FIELD"
# اگر ستون Field_no داری:
# join_left = "Field_no"
join_left = "FIELD"  # <-- عوض کن به ستون درست shapefile
join_right = "Field_no"

In [ ]:
# مطمئن شو نوع‌ها یکی هستند
fields_gdf[join_left] = fields_gdf[join_left].astype(int)
pred[join_right] = pred[join_right].astype(int)

In [ ]:
# Join
g = fields_gdf.merge(pred, left_on=join_left, right_on=join_right, how="left")

In [ ]:
print("Fields:", len(fields_gdf), "Pred joined:", g["som_pred"].notna().sum())

In [ ]:
# ---- رسم نقشه ----
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

In [ ]:
# زمینه (فیلدهایی که prediction ندارند)
g.plot(ax=ax, color="lightgrey", edgecolor="white", linewidth=0.3)

In [ ]:
# فیلدهای دارای prediction
g.dropna(subset=["som_pred"]).plot(
    ax=ax, column="som_pred", legend=True, edgecolor="black", linewidth=0.2
)

In [ ]:
ax.set_title("Predicted SOM (Field-level)", fontsize=14)
ax.set_axis_off()
plt.tight_layout()

In [ ]:
plt.show()
# نقشه تعاملی که میتونی روش زوم کنی و اطلاعات هر مزرعه رو ببینی
g.explore(column="som_pred", cmap="YlOrBr", tiles="CartoDB positron")

In [ ]:
import geopandas as gpd

In [ ]:
samples = gpd.read_file("/content/samples.shp")  # مسیر واقعی
print(samples.columns)
samples = samples.to_crs(g.crs)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

In [ ]:
# 1️⃣ زمینه: همه فیلدها خاکستری
g.plot(ax=ax, color="lightgrey", edgecolor="white", linewidth=0.3)

In [ ]:
# 2️⃣ فیلدهای دارای SOM prediction
g.dropna(subset=["som_pred"]).plot(
    ax=ax,
    column="som_pred",
    cmap="YlOrBr",
    legend=True,
    edgecolor="black",
    linewidth=0.3,
    legend_kwds={"label": "Predicted SOM (%)", "shrink": 0.6},
)

In [ ]:
# 3️⃣ نقاط نمونه خاک (با رنگ مشکی یا بر اساس SOM مشاهده‌ای)
samples.plot(ax=ax, color="black", markersize=20, alpha=0.8, label="Soil samples")

In [ ]:
ax.set_title("Field-level SOM Prediction with Soil Sampling Points", fontsize=14)
ax.set_axis_off()

In [ ]:
plt.legend(loc="lower left")
plt.tight_layout()
samples.plot(
    ax=ax,
    column="Om_p",
    cmap="YlOrBr",
    markersize=30,
    edgecolor="black",
    linewidth=0.3,
    legend=True,
    legend_kwds={"label": "Observed SOM (%)"},
)
plt.show()

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

g: GeoDataFrame فیلدها که som_pred و FIELD دارد
samples: GeoDataFrame نمونه‌ها که Om_p دارد

In [ ]:
# 1) هماهنگ کردن CRS
samples = samples.to_crs(g.crs)

In [ ]:
# 2) فقط ستون‌های لازم از فیلدها
fields_for_join = g[["FIELD", "som_pred", "som_pred_log", "geometry"]].copy()

In [ ]:
# 3) Spatial join (هر نقطه داخل کدام پلیگون است)
# predicate="within" یا "intersects" (within بهتر است)
samples_joined = gpd.sjoin(samples, fields_for_join, how="left", predicate="within")

In [ ]:
print("Samples total:", len(samples))
print("Samples matched to fields:", samples_joined["FIELD"].notna().sum())

In [ ]:
# پاکسازی: فقط نمونه‌هایی که هم Om_p دارند هم pred دارند
df_scatter = samples_joined.dropna(subset=["Om_p", "som_pred"]).copy()

In [ ]:
x = df_scatter["som_pred"].astype(float)  # predicted SOM (linear)
y = df_scatter["Om_p"].astype(float)  # observed SOM at point (linear)

In [ ]:
# Metrics
r2 = 1 - np.sum((y - x) ** 2) / np.sum((y - y.mean()) ** 2)

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(x, y, alpha=0.7)
plt.plot(
    [min(x.min(), y.min()), max(x.max(), y.max())],
    [min(x.min(), y.min()), max(x.max(), y.max())],
)

In [ ]:
plt.xlabel("Predicted SOM (Field-level)")
plt.ylabel("Observed SOM (Sample points)")
plt.title(
    f"Observed vs Predicted SOM (Samples vs Field Prediction)\nR² (approx): {r2:.3f}"
)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
print("Pred log range:", g["som_pred_log"].min(), "to", g["som_pred_log"].max())
print("Pred linear range:", g["som_pred"].min(), "to", g["som_pred"].max())
print("Obs mean_Om_p range (field):", g["som_obs"].min(), "to", g["som_obs"].max())
print("Obs Om_p range (samples):", samples["Om_p"].min(), "to", samples["Om_p"].max())

In [ ]:
# اگر som_pred = 10**som_pred_log درست باشد، این باید نزدیک صفر شود:
check = np.abs(np.log10(g["som_pred"].astype(float)) - g["som_pred_log"].astype(float))
print("Back-transform check (mean abs error in log space):", np.nanmean(check))

In [ ]:
df_field = g.dropna(subset=["som_pred", "som_obs"]).copy()
r2_field = r2_score(df_field["som_obs"], df_field["som_pred"])
rmse_field = np.sqrt(mean_squared_error(df_field["som_obs"], df_field["som_pred"]))
print("Field-level: R2 =", r2_field, "RMSE =", rmse_field)

In [ ]:
rmse_points = np.sqrt(mean_squared_error(y, x))
mae_points = np.mean(np.abs(y - x))
print("Sample-points vs Field-pred: RMSE =", rmse_points, "MAE =", mae_points)

In [ ]:
# uncertainty per field from multi-image predictions
uncertainty = (
    pred_df.groupby("Field_no")
    .agg(
        som_std=("som_pred", "std"),
        q25=("som_pred", lambda x: np.percentile(x, 25)),
        q75=("som_pred", lambda x: np.percentile(x, 75)),
    )
    .reset_index()
)

In [ ]:
uncertainty["som_iqr"] = uncertainty["q75"] - uncertainty["q25"]

In [ ]:
# join to GeoDataFrame
g = g.merge(
    uncertainty[["Field_no", "som_std", "som_iqr"]],
    left_on="FIELD",
    right_on="Field_no",
    how="left",
)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

In [ ]:
g.plot(
    ax=ax,
    column="som_iqr",
    cmap="Purples",
    legend=True,
    edgecolor="black",
    linewidth=0.3,
    legend_kwds={"label": "SOM Uncertainty (IQR)"},
)

In [ ]:
ax.set_title("Uncertainty of Field-level SOM Prediction (IQR across images)")
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# absolute error
g["abs_error"] = np.abs(g["som_pred"] - g["som_obs"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

In [ ]:
# Observed SOM
g.plot(
    ax=axes[0],
    column="som_obs",
    cmap="YlOrBr",
    legend=True,
    edgecolor="black",
    linewidth=0.3,
)
axes[0].set_title("Observed SOM (Field Mean)")
axes[0].set_axis_off()

In [ ]:
# Predicted SOM
g.plot(
    ax=axes[1],
    column="som_pred",
    cmap="YlOrBr",
    legend=True,
    edgecolor="black",
    linewidth=0.3,
)
axes[1].set_title("Predicted SOM (Field-level)")
axes[1].set_axis_off()

In [ ]:
plt.tight_layout()
plt.show()

In [ ]:
threshold = g["abs_error"].quantile(0.80)
g["high_error"] = g["abs_error"] >= threshold

In [ ]:
print("High-error fields:", g["high_error"].sum())

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

In [ ]:
# base map
g.plot(ax=ax, color="lightgrey", edgecolor="white", linewidth=0.3)

In [ ]:
# predicted SOM
g.plot(
    ax=ax, column="som_pred", cmap="YlOrBr", edgecolor="black", linewidth=0.3, alpha=0.7
)

In [ ]:
# highlight high-error fields
g[g["high_error"]].plot(
    ax=ax,
    facecolor="none",
    edgecolor="red",
    linewidth=2,
    label="High-error fields (top 20%)",
)

In [ ]:
ax.set_title("Predicted SOM with High-error Fields Highlighted")
ax.set_axis_off()
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print("pred_df columns:", pred_df.columns.tolist())
print("pred_df head:\n", pred_df.head())
print(
    "unique fields in pred_df:",
    pred_df["Field_no"].nunique() if "Field_no" in pred_df.columns else "NO Field_no",
)
print(
    "unique images in pred_df:",
    pred_df["Image_id"].nunique() if "Image_id" in pred_df.columns else "NO Image_id",
)
print("g columns:", g.columns.tolist())

In [ ]:
import os

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# =========================================================
# CONFIG  ✅ (edit if needed)
# =========================================================
FIELDS_SHP_PATH = "/content/Field_boundary.shp"  # field polygons
PRED_TABLE_PATH = "/content/som_field_predictions.csv"  # optional (field-level table)
# If you already have pred_df in memory, you can skip reading it from file.

In [ ]:
# Column names in field polygons
FIELD_KEY_COL = "FIELD"  # field id in shapefile attributes (must exist)

In [ ]:
# Column names in your prediction tables/dataframes
PRED_DF_FIELD_COL = "Field_no"  # field id in pred_df
PRED_DF_IMG_COL = "Image_id"
PRED_DF_PRED_COL = "som_pred"  # row-level prediction (linear scale) per Field×Image

In [ ]:
# Field-level columns (in g or to be created)
PRED_FIELD_COL = "som_pred"  # field-level prediction (linear)
OBS_FIELD_COL = "som_obs"  # field-level observed (linear)

In [ ]:
TOP_FRAC = 0.20  # highlight worst 20% fields by abs error

In [ ]:
# =========================================================
# 0) Load field polygons (fix SHX if missing)
# =========================================================
os.environ["SHAPE_RESTORE_SHX"] = "YES"
fields_gdf = gpd.read_file(FIELDS_SHP_PATH, engine="fiona")  # fiona is safer for attrs

In [ ]:
print("Fields columns:", fields_gdf.columns.tolist())
if FIELD_KEY_COL not in fields_gdf.columns:
    raise ValueError(
        f"'{FIELD_KEY_COL}' not found in shapefile. Available: {fields_gdf.columns.tolist()}"
    )

In [ ]:
# Ensure join key is numeric
fields_gdf[FIELD_KEY_COL] = pd.to_numeric(
    fields_gdf[FIELD_KEY_COL], errors="coerce"
).astype("Int64")

=========================================================
1) You MUST have pred_df (Field×Image predictions)
   If pred_df already exists in memory, skip this block.
=========================================================
--- If you don't have pred_df, you can build it like this (example):
pred_df = data[['Field_no','Image_id']].copy()
pred_df['som_pred'] = 10 ** rf_final.predict( scaler.transform(data[independent_vars])[:, feat_idx] )
pred_df['mean_Om_p'] = data['mean_Om_p'].values

In [ ]:
# Check pred_df exists
try:
    pred_df
except NameError:
    raise NameError(
        "pred_df is not defined. You need a DataFrame with columns: "
        f"{PRED_DF_FIELD_COL}, {PRED_DF_IMG_COL}, {PRED_DF_PRED_COL} "
        "containing per Field×Image predictions."
    )

In [ ]:
# Validate required columns
required_pred_cols = {PRED_DF_FIELD_COL, PRED_DF_IMG_COL, PRED_DF_PRED_COL}
missing_pred_cols = required_pred_cols - set(pred_df.columns)
if missing_pred_cols:
    raise ValueError(
        f"pred_df missing columns: {missing_pred_cols}. Available: {pred_df.columns.tolist()}"
    )

In [ ]:
# Clean types
tmp = pred_df.copy()
tmp[PRED_DF_FIELD_COL] = pd.to_numeric(tmp[PRED_DF_FIELD_COL], errors="coerce")
tmp[PRED_DF_PRED_COL] = pd.to_numeric(tmp[PRED_DF_PRED_COL], errors="coerce")

In [ ]:
tmp = tmp.dropna(subset=[PRED_DF_FIELD_COL, PRED_DF_PRED_COL])
tmp[PRED_DF_FIELD_COL] = tmp[PRED_DF_FIELD_COL].astype(int)

In [ ]:
print("pred_df rows used:", len(tmp))
print(
    "pred_df unique fields:",
    tmp[PRED_DF_FIELD_COL].nunique(),
    "unique images:",
    tmp[PRED_DF_IMG_COL].nunique(),
)

In [ ]:
# =========================================================
# 2) Build uncertainty per field (ROBUST)
#    - IQR: Q75-Q25
#    - STD: std across images
#    Fields with single image => std NaN -> set 0; iqr -> 0
# =========================================================
def q25(x):
    return np.nanpercentile(x, 25)

In [ ]:
def q75(x):
    return np.nanpercentile(x, 75)

In [ ]:
uncertainty = tmp.groupby(PRED_DF_FIELD_COL, as_index=False).agg(
    som_std=(PRED_DF_PRED_COL, "std"),
    q25=(PRED_DF_PRED_COL, q25),
    q75=(PRED_DF_PRED_COL, q75),
    n_images=(PRED_DF_IMG_COL, "nunique"),
    n_rows=(PRED_DF_PRED_COL, "size"),
)

In [ ]:
uncertainty["som_std"] = uncertainty["som_std"].fillna(0.0)
uncertainty["som_iqr"] = (uncertainty["q75"] - uncertainty["q25"]).fillna(0.0)

In [ ]:
print("Uncertainty table columns:", uncertainty.columns.tolist())
print("Uncertainty head:\n", uncertainty.head())

In [ ]:
# =========================================================
# 3) Build field-level pred/obs table (if not already in g)
#    We'll compute:
#      - som_pred_field = median(pred across images)
#      - som_obs_field  = mean observed (mean_Om_p) if available in pred_df
# =========================================================
field_level = tmp.groupby(PRED_DF_FIELD_COL, as_index=False).agg(
    som_pred=(PRED_DF_PRED_COL, "median"), n_images=(PRED_DF_IMG_COL, "nunique")
)

In [ ]:
# If you have observed field mean in pred_df (e.g., mean_Om_p), add it:
if "mean_Om_p" in tmp.columns:
    obs_level = tmp.groupby(PRED_DF_FIELD_COL, as_index=False).agg(
        som_obs=("mean_Om_p", "mean")
    )
    field_level = field_level.merge(obs_level, on=PRED_DF_FIELD_COL, how="left")
else:
    # create column if not available
    field_level["som_obs"] = np.nan

In [ ]:
# =========================================================
# 4) Join everything to field polygons => g2
# =========================================================
g2 = fields_gdf.merge(
    field_level[[PRED_DF_FIELD_COL, "som_pred", "som_obs", "n_images"]],
    left_on=FIELD_KEY_COL,
    right_on=PRED_DF_FIELD_COL,
    how="left",
)

In [ ]:
g2 = g2.merge(
    uncertainty[[PRED_DF_FIELD_COL, "som_std", "som_iqr"]],
    on=PRED_DF_FIELD_COL,
    how="left",
)

In [ ]:
# Ensure columns exist
for c in ["som_pred", "som_obs", "som_std", "som_iqr", "n_images"]:
    if c not in g2.columns:
        g2[c] = np.nan

In [ ]:
print("Joined fields with som_pred:", g2["som_pred"].notna().sum(), "/", len(g2))
print("Joined fields with som_iqr:", g2["som_iqr"].notna().sum(), "/", len(g2))

In [ ]:
# =========================================================
# 5) (A) Uncertainty maps: IQR + STD
# =========================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

In [ ]:
# background
g2.plot(ax=axes[0], color="lightgrey", edgecolor="white", linewidth=0.3)
g2.dropna(subset=["som_iqr"]).plot(
    ax=axes[0],
    column="som_iqr",
    legend=True,
    edgecolor="black",
    linewidth=0.3,
    cmap="Purples",
    legend_kwds={"label": "Uncertainty (IQR across images)", "shrink": 0.6},
)
axes[0].set_title("SOM Prediction Uncertainty (IQR)")
axes[0].set_axis_off()

In [ ]:
g2.plot(ax=axes[1], color="lightgrey", edgecolor="white", linewidth=0.3)
g2.dropna(subset=["som_std"]).plot(
    ax=axes[1],
    column="som_std",
    legend=True,
    edgecolor="black",
    linewidth=0.3,
    cmap="Blues",
    legend_kwds={"label": "Uncertainty (STD across images)", "shrink": 0.6},
)
axes[1].set_title("SOM Prediction Uncertainty (STD)")
axes[1].set_axis_off()

In [ ]:
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# 6) (B) Observed vs Predicted maps side-by-side
#    Use same vmin/vmax for fair visual comparison
# =========================================================
g_obs_pred = g2.dropna(subset=["som_pred", "som_obs"]).copy()

In [ ]:
if len(g_obs_pred) > 0:
    vmin = float(min(g_obs_pred["som_obs"].min(), g_obs_pred["som_pred"].min()))
    vmax = float(max(g_obs_pred["som_obs"].max(), g_obs_pred["som_pred"].max()))

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    g_obs_pred.plot(
        ax=axes[0],
        column="som_obs",
        legend=True,
        vmin=vmin,
        vmax=vmax,
        cmap="YlOrBr",
        edgecolor="black",
        linewidth=0.3,
        legend_kwds={"label": "Observed SOM (field mean)", "shrink": 0.6},
    )
    axes[0].set_title("Observed SOM")
    axes[0].set_axis_off()

    g_obs_pred.plot(
        ax=axes[1],
        column="som_pred",
        legend=True,
        vmin=vmin,
        vmax=vmax,
        cmap="YlOrBr",
        edgecolor="black",
        linewidth=0.3,
        legend_kwds={"label": "Predicted SOM (field-level)", "shrink": 0.6},
    )
    axes[1].set_title("Predicted SOM")
    axes[1].set_axis_off()

    plt.tight_layout()
    plt.show()
else:
    print(
        "⚠️ Observed vs Predicted map skipped: som_obs is missing (no mean_Om_p in pred_df)."
    )

In [ ]:
# =========================================================
# 7) (C) Highlight highest-error fields (Top TOP_FRAC)
# =========================================================
g_err = g2.copy()
g_err["abs_error"] = np.abs(g_err["som_pred"] - g_err["som_obs"])

In [ ]:
# if som_obs missing, we cannot compute error
g_err_valid = g_err.dropna(subset=["abs_error"]).copy()

In [ ]:
if len(g_err_valid) > 0:
    thr = g_err_valid["abs_error"].quantile(1 - TOP_FRAC)
    g_err_valid["high_error"] = g_err_valid["abs_error"] >= thr

    fig, ax = plt.subplots(1, 1, figsize=(10, 10))

    # base
    g2.plot(ax=ax, color="lightgrey", edgecolor="white", linewidth=0.3)

    # predicted SOM layer
    g2.dropna(subset=["som_pred"]).plot(
        ax=ax,
        column="som_pred",
        legend=True,
        cmap="YlOrBr",
        edgecolor="black",
        linewidth=0.3,
        alpha=0.75,
        legend_kwds={"label": "Predicted SOM", "shrink": 0.6},
    )

    # highlight outlines
    g_err_valid[g_err_valid["high_error"]].plot(
        ax=ax, facecolor="none", edgecolor="red", linewidth=2
    )

    ax.set_title(
        f"Predicted SOM with High-error Fields Highlighted (Top {int(TOP_FRAC*100)}%)"
    )
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()

    # Export worst fields table
    worst_fields = (
        g_err_valid[
            [
                FIELD_KEY_COL,
                "som_pred",
                "som_obs",
                "abs_error",
                "som_iqr",
                "som_std",
                "n_images",
            ]
        ]
        .sort_values("abs_error", ascending=False)
        .head(15)
    )
    print("\nTop 15 worst fields:")
    print(worst_fields)

    worst_fields.to_csv("/content/worst_fields_by_error.csv", index=False)
    print("\nSaved: /content/worst_fields_by_error.csv")

In [ ]:
else:
    print(
        "⚠️ High-error map skipped: som_obs is missing, so abs_error cannot be computed."
    )